# 🛠️ Phase 1 — Environment & Toolchain Setup

Before OptiForge can transform and optimize code, it needs a reliable
execution environment.

OptiForge supports three languages in the MVP:

- 🐍 Python
- ☕ Java
- ⚙️ C++

Each language requires a different execution strategy.

### Python
Python code is executed directly using the Python interpreter.

### Java
Java source code is compiled using `javac` and then executed using `java`.

### C++
C++ source code is compiled using `g++` and then executed as a Windows executable.

Our goal in this phase is to verify that all three toolchains are available.

In [54]:
import sys
import shutil

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

print("\nPython available at:")
print(shutil.which("python"))

Python executable:
d:\LLMprojects\llm_engineering\.venv\Scripts\python.exe

Python version:
3.12.12 (main, Feb 12 2026, 00:40:26) [MSC v.1944 64 bit (AMD64)]

Python available at:
d:\LLMprojects\llm_engineering\.venv\Scripts\python.EXE


## ⚙️ Checking the C++ Toolchain

OptiForge will use GCC for C++ compilation.

We use:

g++ -std=c++17 -O2

where:

- `-std=c++17` → C++17 standard
- `-O2` → compiler optimization
- `-o main.exe` → creates the executable

This is important because OptiForge will eventually generate C++ code
and automatically compile and benchmark it.

In [55]:
import shutil
import subprocess

gpp = shutil.which("g++")

print("g++ path:", gpp)

if gpp:
    result = subprocess.run(
        ["g++", "--version"],
        capture_output=True,
        text=True
    )

    print("\nCompiler version:")
    print(result.stdout.splitlines()[0])
else:
    print("❌ g++ was not found.")

g++ path: C:\msys64\ucrt64\bin\g++.EXE

Compiler version:
g++ (Rev3, Built by MSYS2 project) 16.2.0


## ☕ Checking the Java Toolchain

Java requires two commands:

### javac
Compiles Java source code into bytecode.

### java
Runs the compiled Java program.

OptiForge will eventually use:

javac Main.java
java Main

In [56]:
java = shutil.which("java")
javac = shutil.which("javac")

print("java:", java)
print("javac:", javac)

if java:
    result = subprocess.run(
        ["java", "-version"],
        capture_output=True,
        text=True
    )

    print("\nJava version:")
    print(result.stderr)

java: C:\Program Files\Common Files\Oracle\Java\javapath\java.EXE
javac: C:\Program Files\Common Files\Oracle\Java\javapath\javac.EXE



Java version:
java version "21.0.8" 2025-07-15 LTS
Java(TM) SE Runtime Environment (build 21.0.8+12-LTS-250)
Java HotSpot(TM) 64-Bit Server VM (build 21.0.8+12-LTS-250, mixed mode, sharing)



## 🧪 C++ Compilation Test

Before building the real execution engine, let's prove that OptiForge can:

1. Create C++ source code
2. Compile it
3. Produce an executable
4. Run the executable
5. Capture its output

This will become the foundation of our C++ execution engine.

In [57]:
cpp_code = """
#include <iostream>
using namespace std;

int main() {
    cout << "Hello from OptiForge!" << endl;
    return 0;
}
"""

with open("optiforge_test.cpp", "w", encoding="utf-8") as f:
    f.write(cpp_code)

print("Created optiforge_test.cpp")

Created optiforge_test.cpp


In [58]:
compile_result = subprocess.run(
    [
        "g++",
        "-std=c++17",
        "-O2",
        "optiforge_test.cpp",
        "-o",
        "optiforge_test.exe"
    ],
    capture_output=True,
    text=True
)

if compile_result.returncode == 0:
    print("✅ C++ compilation successful!")
else:
    print("❌ Compilation failed:")
    print(compile_result.stderr)

✅ C++ compilation successful!


In [59]:
run_result = subprocess.run(
    ["optiforge_test.exe"],
    capture_output=True,
    text=True
)

print("Return code:", run_result.returncode)
print("Output:", run_result.stdout)

Return code: 0
Output: Hello from OptiForge!



## 🧪 Java Compilation Test

Now we verify the Java pipeline:

Python → create Java source → javac → java → capture output

In [60]:
java_code = """
public class Main {
    public static void main(String[] args) {
        System.out.println("Hello from OptiForge!");
    }
}
"""

with open("Main.java", "w", encoding="utf-8") as f:
    f.write(java_code)

print("Created Main.java")

Created Main.java


In [61]:
compile_result = subprocess.run(
    ["javac", "Main.java"],
    capture_output=True,
    text=True
)

if compile_result.returncode == 0:
    print("✅ Java compilation successful!")
else:
    print("❌ Compilation failed:")
    print(compile_result.stderr)

✅ Java compilation successful!


In [62]:
run_result = subprocess.run(
    ["java", "Main"],
    capture_output=True,
    text=True
)

print("Return code:", run_result.returncode)
print("Output:", run_result.stdout)

Return code: 0
Output: Hello from OptiForge!



In [63]:
import os

for filename in [
    "optiforge_test.cpp",
    "optiforge_test.exe",
    "Main.java",
    "Main.class"
]:
    if os.path.exists(filename):
        os.remove(filename)

print("🧹 Temporary test files removed.")

🧹 Temporary test files removed.


# 📥 Phase 2 — Code Input & Language Detection

OptiForge needs to accept source code before it can transform or optimize it.

The user should be able to provide code in two ways:

1. 📝 Paste code directly into the application
2. 📄 Upload a source-code file

Supported source languages for the MVP:

- 🐍 Python → `.py`
- ☕ Java → `.java`
- ⚙️ C++ → `.cpp`, `.cc`, `.cxx`

The input pipeline will:

1. Receive the source code
2. Read the file when necessary
3. Detect the programming language
4. Validate the input
5. Return a normalized representation

This normalized representation will be used by the later
transformation, compilation, verification, and benchmarking stages.

## 🌐 Supported Languages

We will keep the language definitions centralized.

This is important because later the execution engine will use the
same language information to decide which compiler or interpreter
should be used.

In [64]:
SUPPORTED_LANGUAGES = {
    "python": {
        "extensions": [".py"],
        "name": "Python"
    },
    "java": {
        "extensions": [".java"],
        "name": "Java"
    },
    "cpp": {
        "extensions": [".cpp", ".cc", ".cxx"],
        "name": "C++"
    }
}

SUPPORTED_LANGUAGES

{'python': {'extensions': ['.py'], 'name': 'Python'},
 'java': {'extensions': ['.java'], 'name': 'Java'},
 'cpp': {'extensions': ['.cpp', '.cc', '.cxx'], 'name': 'C++'}}

## 🔍 Language Detection — File Extension

The most reliable first method is file extension detection.

Examples:

example.py   → Python
Main.java    → Java
solution.cpp → C++
solution.cc  → C++

For the MVP, we will use the extension first instead of trying to
guess the language from the source code itself.

In [65]:
from pathlib import Path


def detect_language_from_extension(filename):
    extension = Path(filename).suffix.lower()

    for language, info in SUPPORTED_LANGUAGES.items():
        if extension in info["extensions"]:
            return language

    return None

In [66]:
test_files = [
    "solution.py",
    "Main.java",
    "program.cpp",
    "algorithm.cc",
    "test.cxx",
    "notes.txt"
]

for filename in test_files:
    print(filename, "→", detect_language_from_extension(filename))

solution.py → python
Main.java → java
program.cpp → cpp
algorithm.cc → cpp
test.cxx → cpp
notes.txt → None


## ✅ Input Validation

OptiForge should reject unsupported languages early.

This prevents problems later when the compiler/execution engine
receives a file it does not know how to execute.

In [67]:
def validate_language(language):
    if language not in SUPPORTED_LANGUAGES:
        raise ValueError(
            f"Unsupported language: {language}"
        )

    return True

In [68]:
validate_language("python")
print("✅ Python is supported")

validate_language("java")
print("✅ Java is supported")

validate_language("cpp")
print("✅ C++ is supported")

✅ Python is supported
✅ Java is supported
✅ C++ is supported


In [69]:
validate_language("javascript")  ## don't woory it will give error😁

ValueError: Unsupported language: javascript

## 📄 Reading Source Code

Once the language is known, OptiForge needs the actual source code.

We will use UTF-8 encoding because it is the standard encoding we
want to support for source files.

In [70]:
def read_source_file(filepath):
    path = Path(filepath)

    if not path.exists():
        raise FileNotFoundError(
            f"File not found: {filepath}"
        )

    if not path.is_file():
        raise ValueError(
            f"Not a file: {filepath}"
        )

    code = path.read_text(encoding="utf-8")

    return code

## 📦 Unified Code Input

The rest of OptiForge should not care whether the user:

- uploaded a file
- pasted source code
- selected a language manually

We therefore normalize everything into one structure.

Example:

{
    "source_code": "...",
    "source_language": "python",
    "filename": "solution.py"
}

This becomes the standard input format for the next phases.

In [71]:
def prepare_source(
    code=None,
    filename=None,
    language="auto"
):
    # Read code from file if provided
    if filename is not None:
        code = read_source_file(filename)

    # Code must exist
    if code is None or not code.strip():
        raise ValueError("No source code provided.")

    # Automatic language detection
    if language == "auto":
        if filename is None:
            raise ValueError(
                "Language cannot be detected automatically "
                "without a filename."
            )

        detected_language = detect_language_from_extension(filename)

        if detected_language is None:
            raise ValueError(
                f"Unsupported file type: {filename}"
            )

        language = detected_language

    # Validate language
    validate_language(language)

    return {
        "source_code": code,
        "source_language": language,
        "filename": filename
    }

## 🧪 Testing the Unified Input

Let's simulate a user pasting Python code.

No file is required when the user explicitly provides the language.

In [72]:
python_code = """
def add(a, b):
    return a + b

print(add(10, 20))
"""

source = prepare_source(
    code=python_code,
    language="python"
)

source

{'source_code': '\ndef add(a, b):\n    return a + b\n\nprint(add(10, 20))\n',
 'source_language': 'python',
 'filename': None}

In [73]:
sample_cpp = """
#include <iostream>

int main() {
    std::cout << "Hello OptiForge!" << std::endl;
    return 0;
}
"""

Path("sample.cpp").write_text(
    sample_cpp,
    encoding="utf-8"
)

101

In [74]:
source = prepare_source(
    filename="sample.cpp",
    language="auto"
)

print("Language:", source["source_language"])
print("Filename:", source["filename"])
print("\nCode:")
print(source["source_code"])

Language: cpp
Filename: sample.cpp

Code:

#include <iostream>

int main() {
    std::cout << "Hello OptiForge!" << std::endl;
    return 0;
}



## Hurray 🎉 We have automatic language detection.

## 🧠 Content-Based Detection

File extensions are preferred because they are reliable.

However, pasted code has no extension.

For pasted code, we can use a lightweight heuristic detector.

This is NOT intended to replace the LLM or a full parser.

It simply provides a reasonable fallback for obvious cases.

In [75]:
def detect_language_from_code(code):
    text = code.lower()

    # Python indicators
    if (
        "def " in text
        or "import " in text
        or "print(" in text
        or "elif " in text
    ):
        return "python"

    # Java indicators
    if (
        "public class " in text
        or "public static void main" in text
        or "system.out.println" in text
    ):
        return "java"

    # C++ indicators
    if (
        "#include <iostream>" in text
        or "#include <bits/stdc++.h>" in text
        or "std::" in text
        or "cout <<" in text
    ):
        return "cpp"

    return None

In [76]:
examples = {
    "Python": """
def hello():
    print("Hello")
""",

    "Java": """
public class Main {
    public static void main(String[] args) {
        System.out.println("Hello");
    }
}
""",

    "C++": """
#include <iostream>

int main() {
    std::cout << "Hello";
}
"""
}

for name, code in examples.items():
    print(name, "→", detect_language_from_code(code))

Python → python
Java → java
C++ → cpp


In [77]:
def prepare_source(
    code=None,
    filename=None,
    language="auto"
):
    # Read file
    if filename is not None:
        code = read_source_file(filename)

    # Validate code
    if code is None or not code.strip():
        raise ValueError("No source code provided.")

    # Automatic detection
    if language == "auto":

        # First preference: file extension
        if filename is not None:
            detected_language = detect_language_from_extension(
                filename
            )

        # Second preference: source-code heuristics
        else:
            detected_language = detect_language_from_code(
                code
            )

        if detected_language is None:
            raise ValueError(
                "Could not determine the source language."
            )

        language = detected_language

    validate_language(language)

    return {
        "source_code": code,
        "source_language": language,
        "filename": filename
    }

In [78]:
source = prepare_source(
    code="""
def square(x):
    return x * x

print(square(5))
""",
    language="auto"
)

print(source["source_language"])

python


In [79]:
source = prepare_source(
    filename="sample.cpp",
    language="auto"
)

print(source["source_language"])

cpp


In [80]:
if Path("sample.cpp").exists():
    Path("sample.cpp").unlink()

print("🧹 Temporary file removed.")

🧹 Temporary file removed.


# 🔍 Phase 3 — Source Code Analysis

Before asking an LLM to transform code, OptiForge performs a lightweight
static analysis of the source program.

The purpose of this analysis is to understand the structure and potential
performance characteristics of the program.

The MVP analysis will collect:

- 📏 Lines of code
- 🔧 Functions / methods
- 🔁 Loops
- 🧩 Imports / includes
- 🖨️ Input/output operations
- 🧠 Complexity hints
- ⚠️ Potential performance-sensitive patterns

The result will be stored as a Code Profile.

This profile will later be passed to the transformation engine so that
the LLM understands what it is optimizing.

In [81]:
from dataclasses import dataclass, field


@dataclass
class CodeProfile:
    language: str
    lines_of_code: int = 0
    functions: list = field(default_factory=list)
    loops: int = 0
    imports: list = field(default_factory=list)
    io_operations: list = field(default_factory=list)
    complexity_hints: list = field(default_factory=list)
    performance_patterns: list = field(default_factory=list)

In [82]:
profile = CodeProfile(language="python")

profile

CodeProfile(language='python', lines_of_code=0, functions=[], loops=0, imports=[], io_operations=[], complexity_hints=[], performance_patterns=[])

## 📏 Source Code Statistics

OptiForge separates source-code lines into four categories:

- 📄 Total lines
- 💻 Actual code lines
- 💬 Comment lines
- ⬜ Blank lines

Comments and blank lines should not be treated as executable code.

This distinction is important because a program containing extensive
documentation should not appear artificially more complex simply because
it has more comments.

The MVP supports common comment styles for:

- Python: `#`
- Java: `//`, `/* ... */`
- C++: `//`, `/* ... */`

In [83]:
def count_lines(code, language):
    lines = code.splitlines()

    total = len(lines)
    blank = 0
    comments = 0
    code_lines = 0

    in_block_comment = False

    for line in lines:
        stripped = line.strip()

        # Blank line
        if not stripped:
            blank += 1
            continue

        # Inside /* ... */ comment
        if in_block_comment:
            comments += 1

            if "*/" in stripped:
                in_block_comment = False

            continue

        # Python comments
        if language == "python":
            if stripped.startswith("#"):
                comments += 1
                continue

        # Java / C++ comments
        if language in ["java", "cpp"]:

            if stripped.startswith("//"):
                comments += 1
                continue

            if stripped.startswith("/*"):
                comments += 1

                if "*/" not in stripped:
                    in_block_comment = True

                continue

        # Everything else is considered code
        code_lines += 1

    return {
        "total": total,
        "code": code_lines,
        "comments": comments,
        "blank": blank
    }

In [84]:
python_test = """
# Calculate the sum

def add(a, b):
    # Return the result
    return a + b

print(add(10, 20))
"""

count_lines(python_test, "python")

{'total': 8, 'code': 3, 'comments': 2, 'blank': 3}

In [85]:
cpp_test = """
// C++ example

#include <iostream>

/*
   Main function
*/
int main() {
    // Print result
    std::cout << "Hello";
    return 0;
}
"""

count_lines(cpp_test, "cpp")

{'total': 13, 'code': 5, 'comments': 5, 'blank': 3}

In [86]:
java_test = """
// Java example

public class Main {

    /*
       Main method
    */
    public static void main(String[] args) {
        // Print result
        System.out.println("Hello");
    }
}
"""

count_lines(java_test, "java")

{'total': 13, 'code': 5, 'comments': 5, 'blank': 3}

## 🔧 Detecting Functions and Methods

Functions are important optimization targets.

For example:

Python:
    def binary_search(...)

Java:
    public int binarySearch(...)

C++:
    int binarySearch(...)

Later, OptiForge can use this information to identify important
parts of a program and ask the LLM to focus on them.

In [87]:
import re

In [88]:
def detect_functions(code, language):
    functions = []

    if language == "python":
        pattern = r"^\s*def\s+([A-Za-z_]\w*)\s*\("

    elif language == "java":
        pattern = (
            r"(?:public|private|protected|static|\s)+"
            r"(?:[\w<>\[\]]+)\s+"
            r"([A-Za-z_]\w*)\s*\("
        )

    elif language == "cpp":
        pattern = (
            r"^\s*(?:[\w:<>,*&]+\s+)+"
            r"([A-Za-z_]\w*)\s*\([^;]*\)\s*\{"
        )

    else:
        return functions

    for line in code.splitlines():
        match = re.search(pattern, line)

        if match:
            functions.append(match.group(1))

    return functions

In [89]:
python_code = """
def calculate_sum(values):
    return sum(values)

def find_max(values):
    return max(values)

print(calculate_sum([1, 2, 3]))
"""

detect_functions(python_code, "python")

['calculate_sum', 'find_max']

## 🔁 Detecting Loops

Loops are one of the first places we look for potential performance
bottlenecks.

Examples:

Python:
    for x in values:
    while condition:

Java:
    for (...)
    while (...)

C++:
    for (...)
    while (...)

The MVP will count loop occurrences.

Later we can perform deeper analysis such as:

    O(n)
    O(n²)
    O(n³)

and identify nested loops.

In [90]:
def count_loops(code, language):
    patterns = {
        "python": [
            r"\bfor\b",
            r"\bwhile\b"
        ],
        "java": [
            r"\bfor\s*\(",
            r"\bwhile\s*\("
        ],
        "cpp": [
            r"\bfor\s*\(",
            r"\bwhile\s*\("
        ]
    }

    if language not in patterns:
        return 0

    count = 0

    for pattern in patterns[language]:
        count += len(re.findall(pattern, code))

    return count

In [91]:
loop_code = """
for i in range(100):
    for j in range(100):
        print(i, j)
"""

count_loops(loop_code, "python")

2

## 🧩 Detecting Dependencies

OptiForge should know which standard libraries or modules the program
uses.

Examples:

Python:
    import math
    from collections import deque

Java:
    import java.util.*;

C++:
    #include <iostream>
    #include <vector>

This information will become part of the Code Profile.

In [92]:
def detect_imports(code, language):
    imports = []

    if language == "python":
        patterns = [
            r"^\s*import\s+(.+)",
            r"^\s*from\s+(.+?)\s+import\s+(.+)"
        ]

    elif language == "java":
        patterns = [
            r"^\s*import\s+(.+);"
        ]

    elif language == "cpp":
        patterns = [
            r'^\s*#include\s*[<"]([^>"]+)[>"]'
        ]

    else:
        return imports

    for line in code.splitlines():
        for pattern in patterns:
            match = re.search(pattern, line)

            if match:
                imports.append(match.group(0).strip())

    return imports

In [93]:
cpp_code = """
#include <iostream>
#include <vector>
#include <algorithm>

int main() {
    return 0;
}
"""

detect_imports(cpp_code, "cpp")

['#include <iostream>', '#include <vector>', '#include <algorithm>']

## 🖨️ Detecting Input and Output

OptiForge needs to know whether a program interacts with standard input
or produces standard output.

Examples:

Python:
    input()
    print()

Java:
    Scanner
    System.out.println()

C++:
    cin
    cout

This will later help us design test cases and compare program outputs.

In [94]:
def detect_io(code, language):
    operations = []

    patterns = {
        "python": {
            "input": r"\binput\s*\(",
            "output": r"\bprint\s*\("
        },

        "java": {
            "input": r"\bScanner\b|\.next\w*\s*\(",
            "output": r"\bSystem\.out\."
        },

        "cpp": {
            "input": r"\bcin\s*>>",
            "output": r"\bcout\s*<<"
        }
    }

    if language not in patterns:
        return operations

    for operation, pattern in patterns[language].items():

        if re.search(pattern, code):
            operations.append(operation)

    return operations

In [95]:
io_code = """
n = int(input())

for i in range(n):
    print(i)
"""

detect_io(io_code, "python")

['input', 'output']

## 🧠 Complexity Hints

Determining exact Big-O complexity requires deeper program analysis.

The MVP therefore produces heuristic hints.

Examples:

- One loop → possible O(n)
- Nested loops → possible O(n²)
- Three nested loops → possible O(n³)
- Sorting → possible O(n log n)
- Hash-based structures → possible average O(1) lookup

These are hints, not mathematical proofs.

Later versions can use AST analysis and LLM-assisted reasoning
to produce more accurate complexity analysis.

In [96]:
def complexity_hints(code, language):
    hints = []

    loop_count = count_loops(code, language)

    if loop_count == 1:
        hints.append("Possible linear iteration: O(n)")

    elif loop_count == 2:
        hints.append(
            "Multiple loops detected; inspect for nesting and possible O(n²)"
        )

    elif loop_count >= 3:
        hints.append(
            "Several loops detected; inspect for higher-order complexity"
        )

    if re.search(r"\bsort\s*\(", code):
        hints.append(
            "Sorting operation detected: commonly O(n log n)"
        )

    if re.search(
        r"\b(set|dict|map|unordered_map|HashMap|HashSet)\b",
        code
    ):
        hints.append(
            "Hash/map data structure detected"
        )

    return hints

In [97]:
complex_code = """
for i in range(n):
    for j in range(n):
        print(i, j)
"""

complexity_hints(complex_code, "python")

['Multiple loops detected; inspect for nesting and possible O(n²)']

## ⚠️ Performance-Sensitive Patterns

We can identify common patterns that may deserve optimization.

Examples:

- Nested loops
- Repeated function calls inside loops
- Sorting
- Large list construction
- Repeated string concatenation
- Expensive operations inside loops

These are signals for the LLM, not automatic optimization decisions.

The LLM proposes a transformation.
The verifier and benchmark later determine whether it actually helped.

In [98]:
def detect_performance_patterns(code, language):
    patterns = []

    loop_count = count_loops(code, language)

    if loop_count >= 2:
        patterns.append("Multiple loops detected")

    if re.search(r"\bsort\s*\(", code):
        patterns.append("Sorting operation")

    if language == "python":
        if re.search(r"\+\s*=\s*[\"']", code):
            patterns.append(
                "Repeated string concatenation may be expensive"
            )

    if language in ["java", "cpp"]:
        if re.search(r"\.append\s*\(", code):
            patterns.append("Repeated append operation")

    return patterns

## 🧩 Building the Complete Source Analyzer

We now combine the individual analysis functions into one analyzer.

Input:

    source_code
    source_language

Output:

    CodeProfile

This gives the rest of OptiForge a clean interface.

In [99]:
lines_of_code: int = 0

In [100]:
@dataclass
class CodeProfile:
    language: str
    total_lines: int = 0
    code_lines: int = 0
    comment_lines: int = 0
    blank_lines: int = 0
    functions: list = field(default_factory=list)
    loops: int = 0
    imports: list = field(default_factory=list)
    io_operations: list = field(default_factory=list)
    complexity_hints: list = field(default_factory=list)
    performance_patterns: list = field(default_factory=list)

In [101]:
def analyze_code(code, language):
    line_info = count_lines(code, language)

    profile = CodeProfile(
        language=language,
        total_lines=line_info["total"],
        code_lines=line_info["code"],
        comment_lines=line_info["comments"],
        blank_lines=line_info["blank"],
        functions=detect_functions(code, language),
        loops=count_loops(code, language),
        imports=detect_imports(code, language),
        io_operations=detect_io(code, language),
        complexity_hints=complexity_hints(code, language),
        performance_patterns=detect_performance_patterns(
            code,
            language
        )
    )

    return profile

In [102]:
sample_dsa = """
def find_pair(values, target):
    for i in range(len(values)):
        for j in range(i + 1, len(values)):
            if values[i] + values[j] == target:
                return i, j

        # Return -1, -1 if no pair is found
    return -1, -1

values = list(map(int, input().split()))
target = int(input())

result = find_pair(values, target)

print(result)
"""

In [103]:
profile = analyze_code(
    sample_dsa,
    "python"
)

profile

CodeProfile(language='python', total_lines=16, code_lines=10, comment_lines=1, blank_lines=5, functions=['find_pair'], loops=2, imports=[], io_operations=['input', 'output'], complexity_hints=['Multiple loops detected; inspect for nesting and possible O(n²)', 'Hash/map data structure detected'], performance_patterns=['Multiple loops detected'])

In [104]:
def display_profile(profile):
    print("🔍 CODE PROFILE")
    print("=" * 40)

    print(f"Language        : {profile.language}")
    print(f"Total lines     : {profile.total_lines}")
    print(f"Code lines      : {profile.code_lines}")
    print(f"Comment lines   : {profile.comment_lines}")
    print(f"Blank lines     : {profile.blank_lines}")
    print(f"Functions       : {profile.functions}")
    print(f"Loop count      : {profile.loops}")

    print("\nImports:")
    for item in profile.imports:
        print(f"  • {item}")

    print("\nI/O:")
    for item in profile.io_operations:
        print(f"  • {item}")

    print("\nComplexity hints:")
    for item in profile.complexity_hints:
        print(f"  • {item}")

    print("\nPerformance patterns:")
    for item in profile.performance_patterns:
        print(f"  • {item}")

In [105]:
display_profile(profile)

🔍 CODE PROFILE
Language        : python
Total lines     : 16
Code lines      : 10
Comment lines   : 1
Blank lines     : 5
Functions       : ['find_pair']
Loop count      : 2

Imports:

I/O:
  • input
  • output

Complexity hints:
  • Multiple loops detected; inspect for nesting and possible O(n²)
  • Hash/map data structure detected

Performance patterns:
  • Multiple loops detected


# 🤖 Phase 4 — LLM Transformation Engine

The transformation engine is the AI component of OptiForge.

It receives:

- Source code
- Source language
- Target language
- Code analysis profile
- Optimization requirements

and asks an LLM to produce a transformed implementation.

The engine supports two major modes:

### 🔄 Translation
Transform code from one language into another.

Example:

Python → C++

### ⚡ Optimization
Improve code while keeping the same language.

Example:

Python → Python

The generated code is only a candidate.

OptiForge does NOT assume that generated code is correct or faster.

The candidate must later pass:

1. Compilation
2. Execution
3. Correctness verification
4. Performance benchmarking

## 🔄 Transformation Modes

OptiForge determines the transformation mode from the source and
target languages.

If the languages are identical:

    Python → Python
    Java → Java
    C++ → C++

the task is primarily optimization.

If the languages differ:

    Python → C++
    Python → Java
    Java → C++

the task is translation combined with optimization.

The target implementation must preserve the behavior of the original
program.

In [106]:
def get_transformation_mode(source_language, target_language):
    if source_language == target_language:
        return "optimization"

    return "translation_and_optimization"

In [107]:
print(get_transformation_mode("python", "python"))
print(get_transformation_mode("python", "cpp"))
print(get_transformation_mode("java", "cpp"))

optimization
translation_and_optimization
translation_and_optimization


## 🧠 Designing the Transformation Prompt

The system prompt defines the role of the LLM.

The LLM is instructed to act as a software transformation and
performance engineer.

It must generate code, but it must not claim that its optimization
is correct or faster.

Those properties will be determined experimentally by OptiForge.

In [108]:
TRANSFORMATION_SYSTEM_PROMPT = """
You are OptiForge AI, a software transformation and optimization engine.

Your task is to transform source code into the requested target language
while preserving the original program's observable behavior.

Requirements:

1. Preserve the program's intended behavior.
2. Preserve input/output behavior.
3. Preserve important edge cases.
4. Preserve algorithmic correctness.
5. Improve performance where a meaningful optimization is possible.
6. Avoid unnecessary complexity.
7. Use appropriate data structures and algorithms.
8. Produce complete, executable source code.
9. Do not include explanations outside the source code.
10. Do not claim that the generated code is correct or faster.
11. Do not invent external dependencies unless necessary.
12. Prefer standard libraries.

The generated program will be compiled, executed, verified, and benchmarked
by OptiForge after generation.

Return ONLY the complete target-language source code.
"""

In [109]:
def build_transformation_prompt(
    source_code,
    source_language,
    target_language,
    profile
):
    mode = get_transformation_mode(
        source_language,
        target_language
    )

    prompt = f"""
Transformation mode:
{mode}

Source language:
{source_language}

Target language:
{target_language}

Code statistics:
- Total lines: {profile.total_lines}
- Code lines: {profile.code_lines}
- Comment lines: {profile.comment_lines}
- Blank lines: {profile.blank_lines}

Functions:
{profile.functions}

Loop count:
{profile.loops}

Imports:
{profile.imports}

I/O operations:
{profile.io_operations}

Complexity hints:
{profile.complexity_hints}

Performance-sensitive patterns:
{profile.performance_patterns}

SOURCE CODE:
----------------
{source_code}
----------------

Generate the complete {target_language} implementation.
"""

    return prompt

In [110]:
source_code = """
def find_pair(values, target):
    for i in range(len(values)):
        for j in range(i + 1, len(values)):
            if values[i] + values[j] == target:
                return i, j

    return -1, -1
"""

profile = analyze_code(
    source_code,
    "python"
)

prompt = build_transformation_prompt(
    source_code,
    "python",
    "cpp",
    profile
)

print(prompt)


Transformation mode:
translation_and_optimization

Source language:
python

Target language:
cpp

Code statistics:
- Total lines: 8
- Code lines: 6
- Comment lines: 0
- Blank lines: 2

Functions:
['find_pair']

Loop count:
2

Imports:
[]

I/O operations:
[]

Complexity hints:
['Multiple loops detected; inspect for nesting and possible O(n²)']

Performance-sensitive patterns:
['Multiple loops detected']

SOURCE CODE:
----------------

def find_pair(values, target):
    for i in range(len(values)):
        for j in range(i + 1, len(values)):
            if values[i] + values[j] == target:
                return i, j

    return -1, -1

----------------

Generate the complete cpp implementation.



## 🤖 LLM Provider Setup

OptiForge supports multiple LLM providers through OpenAI-compatible APIs.

Each provider is initialized as a client.

The transformation engine will use the `clients` dictionary to select
the appropriate client based on the selected model.

In [111]:
import sys
!{sys.executable} -m pip install openai python-dotenv


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [112]:
import openai
from dotenv import load_dotenv

print("OpenAI:", openai.__version__)
print("Imports successful ✅")
load_dotenv()

OpenAI: 2.7.1
Imports successful ✅


True

In [113]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")



OpenAI API Key exists and begins sk-proj-
Anthropic API Key not set (and this is optional)
Google API Key exists and begins AQ
Grok API Key exists and begins xai-
Groq API Key exists and begins gsk_
OpenRouter API Key exists and begins sk-or-


In [114]:
# Connect to client libraries
from openai import OpenAI

# Connect to client libraries
openai_client = OpenAI() # It's good practice to name the instance something other than the module name

anthropic_url = "https://api.anthropic.com/v1/"
# ... rest of your URL and client initializations
openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
grok_url = "https://api.x.ai/v1"
groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)

In [115]:
models = [
    "qwen3.5:2b",
    "openai/gpt-oss-120b",
    "gemini-2.5-pro",
    "llama3.2:latest",
    "qwen/qwen3-coder-30b-a3b-instruct"
]

In [116]:
clients = {
    "qwen3.5:2b": ollama,
    "openai/gpt-oss-120b": groq,
    "gemini-2.5-pro": gemini,
    "llama3.2:latest": ollama,
    "qwen/qwen3-coder-30b-a3b-instruct": openrouter
}

print("✅ Model clients configured:")
for model, client in clients.items():
    print(f"  • {model}")

✅ Model clients configured:
  • qwen3.5:2b
  • openai/gpt-oss-120b
  • gemini-2.5-pro
  • llama3.2:latest
  • qwen/qwen3-coder-30b-a3b-instruct


In [117]:
print("Ollama:", ollama)
print("Groq:", groq)
print("Gemini:", gemini)
print("OpenRouter:", openrouter)

Ollama: <openai.OpenAI object at 0x000001D6B618BF20>
Groq: <openai.OpenAI object at 0x000001D6B61761B0>
Gemini: <openai.OpenAI object at 0x000001D6B6176900>
OpenRouter: <openai.OpenAI object at 0x000001D6B618B500>


## 🤖 Model Configuration

OptiForge separates the model from the transformation logic.

This allows us to experiment with different LLMs without changing
the rest of the system.

For example:

Qwen → candidate implementation
Gemini → candidate implementation
GPT → candidate implementation

Later, OptiForge can generate multiple candidates and benchmark them.

In [118]:
def generate_code(
    model,
    source_code,
    source_language,
    target_language,
    profile
):
    client = clients[model]

    user_prompt = build_transformation_prompt(
        source_code,
        source_language,
        target_language,
        profile
    )

    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": TRANSFORMATION_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content

Why temperature=0?

For this project we want:

reliability > creativity

We're generating software that will be compiled and tested.

In [119]:
def clean_generated_code(code):
    code = code.strip()

    if code.startswith("```"):
        lines = code.splitlines()

        # Remove opening fence
        lines = lines[1:]

        # Remove closing fence
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]

        code = "\n".join(lines)

    return code.strip()

In [120]:
test_output = """```cpp
#include <iostream>

int main() {
    std::cout << "Hello";
}
```"""

print(clean_generated_code(test_output))

#include <iostream>

int main() {
    std::cout << "Hello";
}


In [121]:
python_code = """
def calculate_sum(values):
    total = 0

    for value in values:
        total += value

    print(total)


values = list(map(int, input().split()))
calculate_sum(values)
"""

In [122]:
profile = analyze_code(
    python_code,
    "python"
)

display_profile(profile)

🔍 CODE PROFILE
Language        : python
Total lines     : 12
Code lines      : 7
Comment lines   : 0
Blank lines     : 5
Functions       : ['calculate_sum']
Loop count      : 1

Imports:

I/O:
  • input
  • output

Complexity hints:
  • Possible linear iteration: O(n)
  • Hash/map data structure detected

Performance patterns:


In [123]:
target_language = "cpp"

generated_code = generate_code(
    model="qwen/qwen3-coder-30b-a3b-instruct",
    source_code=python_code,
    source_language="python",
    target_language=target_language,
    profile=profile
)

generated_code = clean_generated_code(generated_code)

print(generated_code)

#include <iostream>
#include <vector>
#include <sstream>

int main() {
    std::string line;
    std::getline(std::cin, line);
    
    std::istringstream iss(line);
    std::vector<int> values;
    int value;
    
    while (iss >> value) {
        values.push_back(value);
    }
    
    long long total = 0;
    for (const auto& v : values) {
        total += v;
    }
    
    std::cout << total << std::endl;
    
    return 0;
}


In [124]:
'''
                  SOURCE
                    │
                    ▼
             📦 Source Object
                    │
                    ▼
             🔍 Code Profile
                    │
                    ▼
           ┌──────────────────┐
           │  Prompt Builder  │
           └────────┬─────────┘
                    │
                    ▼
                🤖 LLM
                    │
             ┌──────┴──────┐
             │             │
          Python         C++/Java
             │             │
             └──────┬──────┘
                    ▼
             🧹 Output Cleaner
                    │
                    ▼
             🤖 Candidate Code
                    │
                    ▼
              Phase 5 🔜'''

'\n                  SOURCE\n                    │\n                    ▼\n             📦 Source Object\n                    │\n                    ▼\n             🔍 Code Profile\n                    │\n                    ▼\n           ┌──────────────────┐\n           │  Prompt Builder  │\n           └────────┬─────────┘\n                    │\n                    ▼\n                🤖 LLM\n                    │\n             ┌──────┴──────┐\n             │             │\n          Python         C++/Java\n             │             │\n             └──────┬──────┘\n                    ▼\n             🧹 Output Cleaner\n                    │\n                    ▼\n             🤖 Candidate Code\n                    │\n                    ▼\n              Phase 5 🔜'

# 🔧 Phase 5 — Multi-Language Execution Engine

The execution engine is responsible for compiling and running source code.

OptiForge supports:

- 🐍 Python
- ☕ Java
- ⚙️ C++

Each language has a different execution process.

Python:
    Python source → Python interpreter → output

Java:
    Java source → javac → bytecode → java → output

C++:
    C++ source → g++ → executable → output

The execution engine hides these differences behind a common interface.

The rest of OptiForge only needs to ask:

    "Run this code."

and receives a standardized ExecutionResult.

## 📦 Execution Result

Every execution should return structured information.

We need more than just stdout because later phases need to know:

- whether compilation succeeded
- whether execution succeeded
- what the program printed
- whether an error occurred
- the exit code
- how long execution took

In [125]:
from dataclasses import dataclass


@dataclass
class ExecutionResult:
    success: bool
    stdout: str
    stderr: str
    return_code: int
    runtime: float
    phase: str

In [126]:
import time

## 🐍 Python Execution

Python does not require a compilation step for our MVP.

OptiForge will execute Python using the same Python interpreter
running our notebook.

Using `sys.executable` is preferable to simply calling `python`
because it ensures we use the active OptiForge environment.

In [127]:
import subprocess
import sys
import tempfile
from pathlib import Path

In [128]:
def run_python_code(code, input_data=""):
    with tempfile.TemporaryDirectory() as temp_dir:

        source_path = Path(temp_dir) / "main.py"

        source_path.write_text(
            code,
            encoding="utf-8"
        )

        start = time.perf_counter()

        try:
            result = subprocess.run(
                [sys.executable, str(source_path)],
                input=input_data,
                capture_output=True,
                text=True,
                timeout=10
            )

            runtime = time.perf_counter() - start

            return ExecutionResult(
                success=result.returncode == 0,
                stdout=result.stdout,
                stderr=result.stderr,
                return_code=result.returncode,
                runtime=runtime,
                phase="execution"
            )

        except subprocess.TimeoutExpired as e:

            runtime = time.perf_counter() - start

            return ExecutionResult(
                success=False,
                stdout=e.stdout or "",
                stderr="Execution timed out.",
                return_code=-1,
                runtime=runtime,
                phase="execution"
            )

In [129]:
python_test = """
a = 10
b = 20

print(a + b)
"""

result = run_python_code(
    python_test
)

print(result)

ExecutionResult(success=True, stdout='30\n', stderr='', return_code=0, runtime=0.17191639996599406, phase='execution')


## ⚙️ C++ Execution

C++ requires two steps:

1. Compilation
2. Execution

OptiForge will compile using:

    g++ -std=c++17 -O2

The generated executable is then executed.

Compilation and execution errors are captured separately.

In [130]:
def run_cpp_code(code, input_data=""):
    with tempfile.TemporaryDirectory() as temp_dir:

        source_path = Path(temp_dir) / "main.cpp"
        executable_path = Path(temp_dir) / "main.exe"

        source_path.write_text(
            code,
            encoding="utf-8"
        )

        # -------------------------
        # Compilation
        # -------------------------

        compile_result = subprocess.run(
            [
                "g++",
                "-std=c++17",
                "-O2",
                str(source_path),
                "-o",
                str(executable_path)
            ],
            capture_output=True,
            text=True,
            timeout=30
        )

        if compile_result.returncode != 0:

            return ExecutionResult(
                success=False,
                stdout="",
                stderr=compile_result.stderr,
                return_code=compile_result.returncode,
                runtime=0.0,
                phase="compilation"
            )

        # -------------------------
        # Execution
        # -------------------------

        start = time.perf_counter()

        try:

            result = subprocess.run(
                [str(executable_path)],
                input=input_data,
                capture_output=True,
                text=True,
                timeout=10
            )

            runtime = time.perf_counter() - start

            return ExecutionResult(
                success=result.returncode == 0,
                stdout=result.stdout,
                stderr=result.stderr,
                return_code=result.returncode,
                runtime=runtime,
                phase="execution"
            )

        except subprocess.TimeoutExpired:

            runtime = time.perf_counter() - start

            return ExecutionResult(
                success=False,
                stdout="",
                stderr="Execution timed out.",
                return_code=-1,
                runtime=runtime,
                phase="execution"
            )

In [131]:
cpp_test = """
#include <iostream>

int main() {
    int a, b;
    std::cin >> a >> b;

    std::cout << a + b << std::endl;

    return 0;
}
"""

In [132]:
result = run_cpp_code(
    cpp_test,
    input_data="10 20\n"
)

print(result)

ExecutionResult(success=True, stdout='30\n', stderr='', return_code=0, runtime=0.1678978999843821, phase='execution')


## ☕ Java Execution

Java requires compilation before execution.

The execution process is:

    Main.java
       ↓
    javac Main.java
       ↓
    Main.class
       ↓
    java Main

In [133]:
def run_java_code(code, input_data=""):
    with tempfile.TemporaryDirectory() as temp_dir:

        source_path = Path(temp_dir) / "Main.java"

        source_path.write_text(
            code,
            encoding="utf-8"
        )

        # -------------------------
        # Compilation
        # -------------------------

        compile_result = subprocess.run(
            [
                "javac",
                str(source_path)
            ],
            capture_output=True,
            text=True,
            timeout=30
        )

        if compile_result.returncode != 0:

            return ExecutionResult(
                success=False,
                stdout="",
                stderr=compile_result.stderr,
                return_code=compile_result.returncode,
                runtime=0.0,
                phase="compilation"
            )

        # -------------------------
        # Execution
        # -------------------------

        start = time.perf_counter()

        try:

            result = subprocess.run(
                [
                    "java",
                    "-cp",
                    temp_dir,
                    "Main"
                ],
                input=input_data,
                capture_output=True,
                text=True,
                timeout=10
            )

            runtime = time.perf_counter() - start

            return ExecutionResult(
                success=result.returncode == 0,
                stdout=result.stdout,
                stderr=result.stderr,
                return_code=result.returncode,
                runtime=runtime,
                phase="execution"
            )

        except subprocess.TimeoutExpired:

            runtime = time.perf_counter() - start

            return ExecutionResult(
                success=False,
                stdout="",
                stderr="Execution timed out.",
                return_code=-1,
                runtime=runtime,
                phase="execution"
            )

In [134]:
java_test = """
import java.util.*;

public class Main {
    public static void main(String[] args) {
        Scanner sc = new Scanner(System.in);

        int a = sc.nextInt();
        int b = sc.nextInt();

        System.out.println(a + b);
    }
}
"""

In [135]:
result = run_java_code(
    java_test,
    input_data="10 20\n"
)

print(result)

ExecutionResult(success=True, stdout='30\n', stderr='', return_code=0, runtime=0.2856815999839455, phase='execution')


## 🌐 Universal Execution Interface

The rest of OptiForge should not need to know the internal execution
details of each language.

The universal runner selects the correct execution strategy based
on the source language.

In [136]:
EXECUTORS = {
    "python": run_python_code,
    "cpp": run_cpp_code,
    "java": run_java_code
}

In [137]:
def run_code(code, language, input_data=""):
    if language not in EXECUTORS:
        raise ValueError(
            f"No executor available for: {language}"
        )

    executor = EXECUTORS[language]

    return executor(
        code,
        input_data
    )

In [138]:
run_code(code, "python")
run_code(code, "cpp")
run_code(code, "java")

ExecutionResult(success=False, stdout='', stderr='C:\\Users\\ASUS\\AppData\\Local\\Temp\\tmpwst838co\\Main.java:2: error: illegal character: \'#\'\n#include <iostream>\n^\nC:\\Users\\ASUS\\AppData\\Local\\Temp\\tmpwst838co\\Main.java:5: error: not a statement\n    std::cout << "Hello";\n              ^\nC:\\Users\\ASUS\\AppData\\Local\\Temp\\tmpwst838co\\Main.java:2: error: unnamed classes are a preview feature and are disabled by default.\n#include <iostream>\n         ^\n  (use --enable-preview to enable unnamed classes)\n3 errors\n', return_code=1, runtime=0.0, phase='compilation')

In [139]:
python_code = """
a, b = map(int, input().split())
print(a + b)
"""

python_result = run_code(
    python_code,
    "python",
    "10 20\n"
)

print(python_result)

ExecutionResult(success=True, stdout='30\n', stderr='', return_code=0, runtime=0.1943324999883771, phase='execution')


In [140]:
cpp_code = """
#include <iostream>

int main() {
    int a, b;
    std::cin >> a >> b;

    std::cout << a + b << std::endl;

    return 0;
}
"""

cpp_result = run_code(
    cpp_code,
    "cpp",
    "10 20\n"
)

print(cpp_result)

ExecutionResult(success=True, stdout='30\n', stderr='', return_code=0, runtime=0.15874280000571162, phase='execution')


In [141]:
java_code = """
import java.util.*;

public class Main {
    public static void main(String[] args) {
        Scanner sc = new Scanner(System.in);

        int a = sc.nextInt();
        int b = sc.nextInt();

        System.out.println(a + b);
    }
}
"""

java_result = run_code(
    java_code,
    "java",
    "10 20\n"
)

print(java_result)

ExecutionResult(success=True, stdout='30\n', stderr='', return_code=0, runtime=0.29894630005583167, phase='execution')


In [142]:
def display_execution_result(result):
    print("🔧 EXECUTION RESULT")
    print("=" * 40)

    if result.success:
        print("Status       : ✅ SUCCESS")
    else:
        print("Status       : ❌ FAILED")

    print(f"Phase        : {result.phase}")
    print(f"Return code  : {result.return_code}")
    print(f"Runtime      : {result.runtime:.6f} seconds")

    print("\nSTDOUT:")
    print(result.stdout if result.stdout else "(empty)")

    if result.stderr:
        print("\nSTDERR:")
        print(result.stderr)

In [143]:
display_execution_result(python_result)

🔧 EXECUTION RESULT
Status       : ✅ SUCCESS
Phase        : execution
Return code  : 0
Runtime      : 0.194332 seconds

STDOUT:
30



In [144]:
display_execution_result(cpp_result)

🔧 EXECUTION RESULT
Status       : ✅ SUCCESS
Phase        : execution
Return code  : 0
Runtime      : 0.158743 seconds

STDOUT:
30



In [145]:
display_execution_result(java_result)

🔧 EXECUTION RESULT
Status       : ✅ SUCCESS
Phase        : execution
Return code  : 0
Runtime      : 0.298946 seconds

STDOUT:
30



In [146]:
bad_cpp = """
#include <iostream>

int main() {
    this_is_not_valid_cpp;
    return 0;
}
"""

In [147]:
result = run_code(
    bad_cpp,
    "cpp"
)

display_execution_result(result)

🔧 EXECUTION RESULT
Status       : ❌ FAILED
Phase        : compilation
Return code  : 1
Runtime      : 0.000000 seconds

STDOUT:
(empty)

STDERR:
C:\Users\ASUS\AppData\Local\Temp\tmp9w28gpa7\main.cpp: In function 'int main()':
C:\Users\ASUS\AppData\Local\Temp\tmp9w28gpa7\main.cpp:5:5: error: 'this_is_not_valid_cpp' was not declared in this scope
    5 |     this_is_not_valid_cpp;
      |     ^~~~~~~~~~~~~~~~~~~~~



In [148]:
infinite_python = """
while True:
    pass
"""

In [149]:
result = run_code(
    infinite_python,
    "python"
)

display_execution_result(result)

🔧 EXECUTION RESULT
Status       : ❌ FAILED
Phase        : execution
Return code  : -1
Runtime      : 10.018331 seconds

STDOUT:
(empty)

STDERR:
Execution timed out.


# 🧪 PHASE 6 — Correctness Verification

In [150]:
'''
                 Verification Engine
                         │
          ┌──────────────┼──────────────┐
          ↓              ↓              ↓
      Test Input      Original       Generated
          │              │              │
          │           Execute         Execute
          │              │              │
          └──────────────┼──────────────┘
                         ↓
                  Compare Outputs
                         ↓
                ┌────────┴────────┐
                ↓                 ↓
             MATCH             DIFFER
                ↓                 ↓
             ✅ PASS             ❌ FAIL'''

'\n                 Verification Engine\n                         │\n          ┌──────────────┼──────────────┐\n          ↓              ↓              ↓\n      Test Input      Original       Generated\n          │              │              │\n          │           Execute         Execute\n          │              │              │\n          └──────────────┼──────────────┘\n                         ↓\n                  Compare Outputs\n                         ↓\n                ┌────────┴────────┐\n                ↓                 ↓\n             MATCH             DIFFER\n                ↓                 ↓\n             ✅ PASS             ❌ FAIL'

## 🧪 Verification Result

The execution engine tells us whether a program runs successfully.

The verification engine goes one step further:

- Run the original program
- Run the generated program
- Compare their outputs
- Repeat across multiple test cases
- Report whether the transformation preserved behavior

A transformation is considered verified only when all supplied test cases pass.

In [151]:
from dataclasses import dataclass, field


@dataclass
class TestCaseResult:
    test_number: int
    passed: bool
    expected_output: str
    actual_output: str
    original_runtime: float
    generated_runtime: float
    error: str = ""


@dataclass
class VerificationResult:
    verified: bool
    total_tests: int
    passed_tests: int
    failed_tests: int
    test_results: list = field(default_factory=list)
    reason: str = ""

In [152]:
def normalize_output(output):
    return output.strip()

In [153]:
normalize_output("10\n20\n")

'10\n20'

In [154]:
def compare_outputs(expected, actual):
    return normalize_output(expected) == normalize_output(actual)

In [155]:
expected = "30\n"
actual = "30"

print(compare_outputs(expected, actual))

True


In [156]:
def verify_test_case(
    original_code,
    original_language,
    generated_code,
    generated_language,
    input_data="",
    test_number=1
):
    original_result = run_code(
        original_code,
        original_language,
        input_data
    )

    if not original_result.success:
        return TestCaseResult(
            test_number=test_number,
            passed=False,
            expected_output="",
            actual_output="",
            original_runtime=original_result.runtime,
            generated_runtime=0.0,
            error="Original program failed."
        )

    generated_result = run_code(
        generated_code,
        generated_language,
        input_data
    )

    if not generated_result.success:
        return TestCaseResult(
            test_number=test_number,
            passed=False,
            expected_output=original_result.stdout,
            actual_output=generated_result.stdout,
            original_runtime=original_result.runtime,
            generated_runtime=generated_result.runtime,
            error=f"Generated program failed during {generated_result.phase}."
        )

    passed = compare_outputs(
        original_result.stdout,
        generated_result.stdout
    )

    return TestCaseResult(
        test_number=test_number,
        passed=passed,
        expected_output=original_result.stdout,
        actual_output=generated_result.stdout,
        original_runtime=original_result.runtime,
        generated_runtime=generated_result.runtime,
        error="" if passed else "Output mismatch."
    )

In [157]:
def verify_programs(
    original_code,
    original_language,
    generated_code,
    generated_language,
    test_inputs
):
    results = []

    for index, input_data in enumerate(test_inputs, start=1):
        result = verify_test_case(
            original_code=original_code,
            original_language=original_language,
            generated_code=generated_code,
            generated_language=generated_language,
            input_data=input_data,
            test_number=index
        )

        results.append(result)

    passed_tests = sum(
        1 for result in results
        if result.passed
    )

    failed_tests = len(results) - passed_tests

    verified = (
        len(results) > 0
        and failed_tests == 0
    )

    reason = (
        "All test cases passed."
        if verified
        else f"{failed_tests} test case(s) failed."
    )

    return VerificationResult(
        verified=verified,
        total_tests=len(results),
        passed_tests=passed_tests,
        failed_tests=failed_tests,
        test_results=results,
        reason=reason
    )

In [158]:
original_code = """
a, b = map(int, input().split())
print(a + b)
"""

In [159]:
generated_code = """
#include <iostream>
using namespace std;

int main() {
    int a, b;
    cin >> a >> b;
    cout << a + b << endl;
    return 0;
}
"""

In [160]:
test_inputs = [
    "10 20\n",
    "100 200\n",
    "-5 10\n",
    "0 0\n"
]

In [161]:
verification = verify_programs(
    original_code=original_code,
    original_language="python",
    generated_code=generated_code,
    generated_language="cpp",
    test_inputs=test_inputs
)

In [162]:
print(verification)

VerificationResult(verified=True, total_tests=4, passed_tests=4, failed_tests=0, test_results=[TestCaseResult(test_number=1, passed=True, expected_output='30\n', actual_output='30\n', original_runtime=0.1103446000488475, generated_runtime=0.24849010002799332, error=''), TestCaseResult(test_number=2, passed=True, expected_output='300\n', actual_output='300\n', original_runtime=0.19341250008437783, generated_runtime=0.14651569991838187, error=''), TestCaseResult(test_number=3, passed=True, expected_output='5\n', actual_output='5\n', original_runtime=0.2120545000070706, generated_runtime=0.15988960000686347, error=''), TestCaseResult(test_number=4, passed=True, expected_output='0\n', actual_output='0\n', original_runtime=0.195323699968867, generated_runtime=0.09801319998223335, error='')], reason='All test cases passed.')


In [163]:
def display_verification_result(result):
    print("🧪 VERIFICATION RESULT")
    print("=" * 45)

    if result.verified:
        print("Status       : ✅ VERIFIED")
    else:
        print("Status       : ❌ NOT VERIFIED")

    print(f"Total tests  : {result.total_tests}")
    print(f"Passed       : {result.passed_tests}")
    print(f"Failed       : {result.failed_tests}")

    print("\nTEST DETAILS")
    print("-" * 45)

    for test in result.test_results:
        status = "✅ PASS" if test.passed else "❌ FAIL"

        print(
            f"Test {test.test_number}: {status} | "
            f"Original: {test.original_runtime:.6f}s | "
            f"Generated: {test.generated_runtime:.6f}s"
        )

        if test.error:
            print(f"  Error: {test.error}")

In [164]:
display_verification_result(verification)

🧪 VERIFICATION RESULT
Status       : ✅ VERIFIED
Total tests  : 4
Passed       : 4
Failed       : 0

TEST DETAILS
---------------------------------------------
Test 1: ✅ PASS | Original: 0.110345s | Generated: 0.248490s
Test 2: ✅ PASS | Original: 0.193413s | Generated: 0.146516s
Test 3: ✅ PASS | Original: 0.212055s | Generated: 0.159890s
Test 4: ✅ PASS | Original: 0.195324s | Generated: 0.098013s


# ⚡ Phase 7 — Benchmark Engine

The benchmark engine measures the real execution performance of the
original and transformed programs.

The benchmark process:

1. Execute the original program multiple times.
2. Execute the transformed program multiple times.
3. Collect runtime measurements.
4. Calculate average runtime.
5. Calculate best and worst runtime.
6. Calculate speedup.
7. Calculate percentage improvement.
8. Report whether the transformation improved performance.

The compiler and execution engine provide the measurements.
The benchmark engine performs the statistical comparison.

In [165]:
'''
                 ORIGINAL CODE
                      │
                 ┌────▼────┐
                 │ Execute │
                 └────┬────┘
                      │
              ┌───────▼────────┐
              │  Repeat N times│
              └───────┬────────┘
                      │
                 Runtime data
                      │
                      ▼
                 Statistics
                      │
                      │
                      ▼
                 ┌───────────┐
                 │ Comparison│
                 └─────┬─────┘
                       │
                 ┌─────▼──────┐
                 │  Speedup   │
                 │ Improvement│
                 └────────────┘'''

'\n                 ORIGINAL CODE\n                      │\n                 ┌────▼────┐\n                 │ Execute │\n                 └────┬────┘\n                      │\n              ┌───────▼────────┐\n              │  Repeat N times│\n              └───────┬────────┘\n                      │\n                 Runtime data\n                      │\n                      ▼\n                 Statistics\n                      │\n                      │\n                      ▼\n                 ┌───────────┐\n                 │ Comparison│\n                 └─────┬─────┘\n                       │\n                 ┌─────▼──────┐\n                 │  Speedup   │\n                 │ Improvement│\n                 └────────────┘'

In [166]:
from dataclasses import dataclass, field


@dataclass
class BenchmarkResult:
    original_runs: list = field(default_factory=list)
    generated_runs: list = field(default_factory=list)

    original_average: float = 0.0
    generated_average: float = 0.0

    original_best: float = 0.0
    generated_best: float = 0.0

    original_worst: float = 0.0
    generated_worst: float = 0.0

    speedup: float = 0.0
    improvement_percent: float = 0.0

    faster: bool = False

In [167]:
def calculate_runtime_stats(runtimes):
    if not runtimes:
        return {
            "average": 0.0,
            "best": 0.0,
            "worst": 0.0
        }

    return {
        "average": sum(runtimes) / len(runtimes),
        "best": min(runtimes),
        "worst": max(runtimes)
    }

In [168]:
runtimes = [0.20, 0.18, 0.21, 0.19, 0.20]

calculate_runtime_stats(runtimes)

{'average': 0.196, 'best': 0.18, 'worst': 0.21}

In [169]:
def benchmark_program(
    code,
    language,
    input_data="",
    runs=5
):
    runtimes = []

    for _ in range(runs):
        result = run_code(
            code,
            language,
            input_data
        )

        if not result.success:
            return None, result

        runtimes.append(result.runtime)

    return runtimes, None

In [170]:
def benchmark_comparison(
    original_code,
    original_language,
    generated_code,
    generated_language,
    input_data="",
    runs=5
):
    original_runtimes, original_error = benchmark_program(
        original_code,
        original_language,
        input_data,
        runs
    )

    if original_error:
        raise RuntimeError(
            f"Original program failed: {original_error.stderr}"
        )

    generated_runtimes, generated_error = benchmark_program(
        generated_code,
        generated_language,
        input_data,
        runs
    )

    if generated_error:
        raise RuntimeError(
            f"Generated program failed: {generated_error.stderr}"
        )

    original_stats = calculate_runtime_stats(
        original_runtimes
    )

    generated_stats = calculate_runtime_stats(
        generated_runtimes
    )

    original_average = original_stats["average"]
    generated_average = generated_stats["average"]

    if generated_average > 0:
        speedup = original_average / generated_average
    else:
        speedup = 0.0

    if original_average > 0:
        improvement = (
            (original_average - generated_average)
            / original_average
        ) * 100
    else:
        improvement = 0.0

    return BenchmarkResult(
        original_runs=original_runtimes,
        generated_runs=generated_runtimes,

        original_average=original_average,
        generated_average=generated_average,

        original_best=original_stats["best"],
        generated_best=generated_stats["best"],

        original_worst=original_stats["worst"],
        generated_worst=generated_stats["worst"],

        speedup=speedup,
        improvement_percent=improvement,

        faster=generated_average < original_average
    )

In [171]:
original_code = """
n = int(input())

total = 0

for i in range(1, n + 1):
    total += i

print(total)
"""

In [172]:
generated_code = """
#include <iostream>
using namespace std;

int main() {
    long long n;
    cin >> n;

    long long total = 0;

    for (long long i = 1; i <= n; ++i) {
        total += i;
    }

    cout << total << endl;

    return 0;
}
"""

In [173]:
input_data = "1000000\n"

In [174]:
benchmark = benchmark_comparison(
    original_code=original_code,
    original_language="python",
    generated_code=generated_code,
    generated_language="cpp",
    input_data=input_data,
    runs=5
)

In [175]:
benchmark

BenchmarkResult(original_runs=[0.25599630002398044, 0.2557634999975562, 0.33073249994777143, 0.3488052999600768, 0.2220320999622345], generated_runs=[0.14283000002615154, 0.11109520005993545, 0.1481043000239879, 0.1202659000409767, 0.145031200023368], original_average=0.2826659399783239, generated_average=0.13346532003488393, original_best=0.2220320999622345, generated_best=0.11109520005993545, original_worst=0.3488052999600768, generated_worst=0.1481043000239879, speedup=2.1178980420115376, improvement_percent=52.78337388469277, faster=True)

In [176]:
def display_benchmark_result(result):
    print("⚡ BENCHMARK RESULT")
    print("=" * 50)

    print("\nORIGINAL")
    print(f"Runs       : {result.original_runs}")
    print(f"Average    : {result.original_average:.6f}s")
    print(f"Best       : {result.original_best:.6f}s")
    print(f"Worst      : {result.original_worst:.6f}s")

    print("\nGENERATED")
    print(f"Runs       : {result.generated_runs}")
    print(f"Average    : {result.generated_average:.6f}s")
    print(f"Best       : {result.generated_best:.6f}s")
    print(f"Worst      : {result.generated_worst:.6f}s")

    print("\nPERFORMANCE")
    print(f"Speedup    : {result.speedup:.2f}x")
    print(
        f"Improvement: "
        f"{result.improvement_percent:.2f}%"
    )

    if result.faster:
        print("\n🏆 Generated implementation is faster.")
    else:
        print("\n⚠️ Generated implementation is not faster.")

In [177]:
display_benchmark_result(benchmark)

⚡ BENCHMARK RESULT

ORIGINAL
Runs       : [0.25599630002398044, 0.2557634999975562, 0.33073249994777143, 0.3488052999600768, 0.2220320999622345]
Average    : 0.282666s
Best       : 0.222032s
Worst      : 0.348805s

GENERATED
Runs       : [0.14283000002615154, 0.11109520005993545, 0.1481043000239879, 0.1202659000409767, 0.145031200023368]
Average    : 0.133465s
Best       : 0.111095s
Worst      : 0.148104s

PERFORMANCE
Speedup    : 2.12x
Improvement: 52.78%

🏆 Generated implementation is faster.


In [178]:
'''
                   LLM
                    │
                    │ proposes
                    ▼
              Generated Code
                    │
                    ▼
               Compiler
                    │
                    ▼
               Execution
                    │
                    ▼
               Benchmark
                    │
                    ▼
              REAL NUMBERS
                    │
                    ▼
            Recommendation'''

'\n                   LLM\n                    │\n                    │ proposes\n                    ▼\n              Generated Code\n                    │\n                    ▼\n               Compiler\n                    │\n                    ▼\n               Execution\n                    │\n                    ▼\n               Benchmark\n                    │\n                    ▼\n              REAL NUMBERS\n                    │\n                    ▼\n            Recommendation'

# 🏆 Phase 8 — Candidate Generation & Ranking

OptiForge does not assume that the first generated implementation is the best.

Instead, it can generate multiple candidates and evaluate each one.

Each candidate contains:

- generated source code
- model used
- transformation type
- compilation status
- verification status
- benchmark results
- performance metrics
- final ranking score

Ranking policy:

1. Reject candidates that do not compile.
2. Reject candidates that fail correctness verification.
3. Compare verified candidates by performance.
4. Prefer simpler implementations when performance is effectively equivalent.
5. Select the best verified candidate.

The benchmark engine supplies actual measurements.
The ranking engine makes the final selection.

In [398]:
from dataclasses import dataclass

@dataclass
class CandidateResult:
    candidate_id: str
    model: str
    source_code: str
    language: str
    compiled: bool = False
    verified: bool = False
    verification: object = None
    benchmark: object = None
    score: float = 0.0
    status: str = "generated"

In [180]:
def generate_candidates(
    source_code,
    source_language,
    target_language,
    profile,
    models
):
    candidates = []

    for index, model in enumerate(models, start=1):

        try:
            generated_code = generate_code(
                model=model,
                source_code=source_code,
                source_language=source_language,
                target_language=target_language,
                profile=profile
            )

            candidates.append(
                CandidateResult(
                    candidate_id=f"C{index}",
                    model=model,
                    source_code=clean_generated_code(
                        generated_code
                    ),
                    language=target_language
                )
            )

        except Exception as e:
            print(
                f"⚠️ Candidate {index} failed: {e}"
            )

    return candidates

In [181]:
models = [
    "qwen3.5:2b",
    "openai/gpt-oss-120b",
    "gemini-2.5-pro",
    "llama3.2:latest",
    "qwen/qwen3-coder-30b-a3b-instruct"
]

In [395]:
def evaluate_candidate(
    candidate,
    original_code,
    original_language,
    test_inputs,
    benchmark_input
):
    # 1. Compilation / initial execution
    initial_result = run_code(
        candidate.source_code,
        candidate.language,
        benchmark_input
    )

    if not initial_result.success:
        candidate.status = "compilation_failed"
        return candidate

    candidate.compiled = True

    # 2. Normalize test cases
    test_cases = create_test_cases(test_inputs)

    # 3. Multi-test verification
    verification = verify_programs(
        original_code=original_code,
        original_language=original_language,
        generated_code=candidate.source_code,
        generated_language=candidate.language,
        test_inputs=test_cases
    )

    candidate.verified = verification.verified
    candidate.verification = verification

    # 4. HARD GATE
    # Never benchmark an incorrect candidate.
    if not candidate.verified:
        candidate.status = "verification_failed"
        return candidate

    # 5. Benchmark only verified candidates
    benchmark = benchmark_comparison(
        original_code=original_code,
        original_language=original_language,
        generated_code=candidate.source_code,
        generated_language=candidate.language,
        input_data=benchmark_input,
        runs=5
    )

    candidate.benchmark = benchmark
    candidate.status = "verified"

    return candidate

In [183]:
def rank_candidates(candidates):
    valid_candidates = [
        candidate
        for candidate in candidates
        if candidate.compiled
        and candidate.verified
        and candidate.benchmark is not None
    ]

    if not valid_candidates:
        return []

    valid_candidates.sort(
        key=lambda candidate:
            candidate.benchmark.generated_average
    )

    for rank, candidate in enumerate(
        valid_candidates,
        start=1
    ):
        candidate.score = 1 / (
            candidate.benchmark.generated_average
            + 1e-9
        )

    return valid_candidates

In [184]:
def candidate_rank_key(candidate):
    if not candidate.compiled:
        return (0, float("inf"))

    if not candidate.verified:
        return (0, float("inf"))

    return (
        1,
        candidate.benchmark.generated_average
    )

In [185]:
def rank_candidates(candidates):
    return sorted(
        candidates,
        key=candidate_rank_key
    )

In [186]:
def display_candidate_ranking(candidates):
    print("🏆 CANDIDATE RANKING")
    print("=" * 70)

    for rank, candidate in enumerate(
        candidates,
        start=1
    ):
        print(f"\n#{rank} — {candidate.candidate_id}")

        print(f"Model       : {candidate.model}")
        print(f"Language    : {candidate.language}")
        print(f"Compiled    : {'✅' if candidate.compiled else '❌'}")
        print(f"Verified    : {'✅' if candidate.verified else '❌'}")
        print(f"Status      : {candidate.status}")

        if candidate.benchmark:
            print(
                f"Runtime     : "
                f"{candidate.benchmark.generated_average:.6f}s"
            )

            print(
                f"Speedup     : "
                f"{candidate.benchmark.speedup:.2f}x"
            )

            print(
                f"Improvement : "
                f"{candidate.benchmark.improvement_percent:.2f}%"
            )

In [187]:
def optimize_with_candidates(
    source_code,
    source_language,
    target_language,
    profile,
    models,
    test_inputs,
    benchmark_input
):
    candidates = generate_candidates(
        source_code=source_code,
        source_language=source_language,
        target_language=target_language,
        profile=profile,
        models=models
    )

    evaluated_candidates = []

    for candidate in candidates:

        print(
            f"\n🔄 Evaluating {candidate.candidate_id}"
            f" ({candidate.model})..."
        )

        evaluated = evaluate_candidate(
            candidate=candidate,
            original_code=source_code,
            original_language=source_language,
            test_inputs=test_inputs,
            benchmark_input=benchmark_input
        )

        evaluated_candidates.append(evaluated)

    ranked = rank_candidates(
        evaluated_candidates
    )

    return ranked

This becomes one of the core functions of OptiForge.

In [188]:
'''
                         USER CODE
                            │
                            ▼
                         ANALYZER
                            │
                            ▼
                    ┌───────────────┐
                    │ 5 LLM MODELS  │
                    └───────┬───────┘
                            │
              ┌─────────────┼─────────────┐
              ▼             ▼             ▼
          Candidate 1   Candidate 2   Candidate 3 ...
              │             │             │
           Compile       Compile       Compile
              │             │             │
           Verify        Verify        Verify
              │             │             │
         ┌────┴────┐  ┌─────┴────┐  ┌─────┴────┐
         │         │  │          │  │          │
         ▼         ▼  ▼          ▼  ▼          ▼
       PASS      FAIL PASS     PASS FAIL      PASS
         │            │                       │
      Benchmark    Benchmark              Benchmark
         │            │                       │
         └────────────┴───────────┬───────────┘
                                   ▼
                             🏆 RANKING
                                   │
                                   ▼
                         BEST VERIFIED CODE'''

'\n                         USER CODE\n                            │\n                            ▼\n                         ANALYZER\n                            │\n                            ▼\n                    ┌───────────────┐\n                    │ 5 LLM MODELS  │\n                    └───────┬───────┘\n                            │\n              ┌─────────────┼─────────────┐\n              ▼             ▼             ▼\n          Candidate 1   Candidate 2   Candidate 3 ...\n              │             │             │\n           Compile       Compile       Compile\n              │             │             │\n           Verify        Verify        Verify\n              │             │             │\n         ┌────┴────┐  ┌─────┴────┐  ┌─────┴────┐\n         │         │  │          │  │          │\n         ▼         ▼  ▼          ▼  ▼          ▼\n       PASS      FAIL PASS     PASS FAIL      PASS\n         │            │                       │\n      Benchmark    Benchmark

In [189]:
'''
Given:

Python source
        ↓
5 generated candidates
        ↓
Compile
        ↓
Verify
        ↓
Benchmark

OptiForge should produce:

╔════════════════════════════════════════════╗
║        🧠 OPTIFORGE RECOMMENDATION         ║
╠════════════════════════════════════════════╣
║ Best Language       : C++                  ║
║ Best Candidate      : C3                   ║
║                                            ║
║ Correctness         : ✅ 100%              ║
║ Time Complexity     : O(n)                 ║
║ Space Complexity    : O(n)                 ║
║                                            ║
║ Original Runtime    : 1.284 sec            ║
║ Optimized Runtime   : 0.327 sec            ║
║ Speedup             : 3.93×                ║
║ Improvement         : 74.5%                ║
║                                            ║
║ Decision            : 🏆 ACCEPT             ║
║                                            ║
║ Why?                                       ║
║ Verified equivalent behavior and          ║
║ significantly lower measured runtime.     ║
╚════════════════════════════════════════════╝
'''

'\nGiven:\n\nPython source\n        ↓\n5 generated candidates\n        ↓\nCompile\n        ↓\nVerify\n        ↓\nBenchmark\n\nOptiForge should produce:\n\n╔════════════════════════════════════════════╗\n║        🧠 OPTIFORGE RECOMMENDATION         ║\n╠════════════════════════════════════════════╣\n║ Best Language       : C++                  ║\n║ Best Candidate      : C3                   ║\n║                                            ║\n║ Correctness         : ✅ 100%              ║\n║ Time Complexity     : O(n)                 ║\n║ Space Complexity    : O(n)                 ║\n║                                            ║\n║ Original Runtime    : 1.284 sec            ║\n║ Optimized Runtime   : 0.327 sec            ║\n║ Speedup             : 3.93×                ║\n║ Improvement         : 74.5%                ║\n║                                            ║\n║ Decision            : 🏆 ACCEPT             ║\n║                                            ║\n║ Why?                         

# 🧠 Phase 9 — Intelligent Recommendation Engine

The recommendation engine converts OptiForge's measured evidence into
an actionable engineering recommendation.

It considers:

- correctness
- compilation success
- runtime performance
- speedup
- performance improvement
- algorithmic complexity
- source/target language
- candidate model
- optimization strategy

Correctness is a hard requirement.

A candidate that fails verification can never be recommended,
even if its measured runtime is lower.

The LLM is used to explain the evidence, not fabricate it.

In [190]:
from dataclasses import dataclass, field


@dataclass
class RecommendationResult:
    decision: str
    recommended_candidate: object = None
    recommended_language: str = ""
    reason: str = ""
    strengths: list = field(default_factory=list)
    warnings: list = field(default_factory=list)
    alternatives: list = field(default_factory=list)

In [191]:
def is_candidate_eligible(candidate):
    return (
        candidate.compiled
        and candidate.verified
        and candidate.benchmark is not None
    )

In [192]:
def get_eligible_candidates(candidates):
    return [
        candidate
        for candidate in candidates
        if is_candidate_eligible(candidate)
    ]

In [193]:
def select_best_candidate(candidates):
    eligible = get_eligible_candidates(candidates)

    if not eligible:
        return None

    return min(
        eligible,
        key=lambda candidate:
            candidate.benchmark.generated_average
    )

In [194]:
def build_recommendation(candidates):
    eligible = get_eligible_candidates(candidates)

    if not eligible:
        return RecommendationResult(
            decision="REJECT",
            reason=(
                "No candidate passed compilation, "
                "verification, and benchmarking."
            )
        )

    best = select_best_candidate(eligible)

    benchmark = best.benchmark

    strengths = []
    warnings = []

    if benchmark.faster:
        strengths.append(
            f"Measured {benchmark.speedup:.2f}x speedup"
        )

    if benchmark.improvement_percent > 0:
        strengths.append(
            f"{benchmark.improvement_percent:.2f}% runtime improvement"
        )

    if len(eligible) > 1:
        alternatives = [
            candidate
            for candidate in eligible
            if candidate.candidate_id != best.candidate_id
        ]
    else:
        alternatives = []

    if benchmark.improvement_percent < 5:
        warnings.append(
            "Performance improvement is relatively small."
        )

    return RecommendationResult(
        decision="ACCEPT",
        recommended_candidate=best,
        recommended_language=best.language,
        reason=(
            "The selected candidate passed verification "
            "and achieved the best measured runtime."
        ),
        strengths=strengths,
        warnings=warnings,
        alternatives=alternatives
    )

In [195]:
MIN_IMPROVEMENT_PERCENT = 5.0

In [196]:
def is_meaningful_improvement(benchmark):
    return (
        benchmark.faster
        and benchmark.improvement_percent
        >= MIN_IMPROVEMENT_PERCENT
    )

In [197]:
'''
                     Candidate
                         │
                         ▼
                    Compiles?
                    /       \
                  NO        YES
                  ↓          ↓
                REJECT    Verified?
                           /     \
                         NO       YES
                         ↓         ↓
                       REJECT   Benchmark
                                    │
                                    ▼
                              Meaningfully faster?
                                /          \
                              NO            YES
                              ↓              ↓
                       KEEP ORIGINAL      ACCEPT'''

'\n                     Candidate\n                         │\n                         ▼\n                    Compiles?\n                    /                         NO        YES\n                  ↓          ↓\n                REJECT    Verified?\n                           /                              NO       YES\n                         ↓         ↓\n                       REJECT   Benchmark\n                                    │\n                                    ▼\n                              Meaningfully faster?\n                                /                                        NO            YES\n                              ↓              ↓\n                       KEEP ORIGINAL      ACCEPT'

In [198]:
@dataclass
class DSAAnalysis:
    algorithm: str = ""
    time_complexity: str = ""
    space_complexity: str = ""
    approach: str = ""
    optimization_opportunity: str = ""

In [199]:
dsa = DSAAnalysis(
    algorithm="Hash Map",
    time_complexity="O(n)",
    space_complexity="O(n)",
    approach="Single-pass lookup",
    optimization_opportunity="Avoid nested search"
)

In [200]:
RECOMMENDATION_SYSTEM_PROMPT = """
You are OptiForge AI's engineering recommendation assistant.

Your job is to explain an optimization decision using only the
provided measured and analyzed evidence.

Rules:

1. Never invent benchmark results.
2. Never claim correctness unless verification says it passed.
3. Never override the verification result.
4. Clearly distinguish measured facts from reasoning.
5. Explain why the selected implementation is preferable.
6. Discuss algorithmic complexity when available.
7. Mention important trade-offs.
8. If the performance improvement is small, say so.
9. If no candidate is verified, recommend rejecting the transformation.
10. Keep the explanation concise and technically useful.

Return a professional engineering recommendation.
"""

In [201]:
def build_recommendation_prompt(
    original_language,
    candidates,
    dsa_analysis=None
):
    candidate_data = []

    for candidate in candidates:
        data = {
            "candidate_id": candidate.candidate_id,
            "model": candidate.model,
            "language": candidate.language,
            "compiled": candidate.compiled,
            "verified": candidate.verified,
            "status": candidate.status
        }

        if candidate.benchmark:
            data.update({
                "average_runtime":
                    candidate.benchmark.generated_average,
                "speedup":
                    candidate.benchmark.speedup,
                "improvement_percent":
                    candidate.benchmark.improvement_percent
            })

        candidate_data.append(data)

    prompt = f"""
Original language:
{original_language}

Candidate evidence:
{candidate_data}

DSA analysis:
{dsa_analysis}

Based only on this evidence:

1. Identify the best verified candidate.
2. Explain why it was selected.
3. Explain the performance trade-offs.
4. Mention algorithmic complexity if available.
5. State whether the optimization should be accepted.
"""

    return prompt

In [202]:
def generate_recommendation_explanation(
    model,
    original_language,
    candidates,
    dsa_analysis=None
):
    client = clients[model]

    prompt = build_recommendation_prompt(
        original_language,
        candidates,
        dsa_analysis
    )

    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": RECOMMENDATION_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content

Final Recommendation Display

In [203]:
def display_recommendation(result):
    print("🧠 OPTIFORGE RECOMMENDATION")
    print("=" * 55)

    print(f"Decision     : {result.decision}")

    if result.recommended_candidate:
        candidate = result.recommended_candidate

        print(
            f"Candidate    : "
            f"{candidate.candidate_id}"
        )

        print(
            f"Model        : "
            f"{candidate.model}"
        )

        print(
            f"Language     : "
            f"{result.recommended_language}"
        )

        if candidate.benchmark:
            benchmark = candidate.benchmark

            print(
                f"\nRuntime      : "
                f"{benchmark.generated_average:.6f}s"
            )

            print(
                f"Speedup      : "
                f"{benchmark.speedup:.2f}x"
            )

            print(
                f"Improvement  : "
                f"{benchmark.improvement_percent:.2f}%"
            )

    print(f"\nReason:")
    print(result.reason)

    if result.strengths:
        print("\nStrengths:")
        for item in result.strengths:
            print(f"  ✅ {item}")

    if result.warnings:
        print("\nWarnings:")
        for item in result.warnings:
            print(f"  ⚠️ {item}")

In [204]:
'''
🚀 The entire OptiForge intelligence pipeline

                         ┌───────────────┐
                         │ SOURCE CODE   │
                         └───────┬───────┘
                                 ↓
                         ┌───────────────┐
                         │   ANALYZER    │
                         └───────┬───────┘
                                 ↓
                         ┌───────────────┐
                         │ LLM GENERATOR │
                         └───────┬───────┘
                                 ↓
                     ┌───────────┼───────────┐
                     ↓           ↓           ↓
                    C1          C2          C3 ...
                     ↓           ↓           ↓
                 COMPILE     COMPILE     COMPILE
                     ↓           ↓           ↓
                  VERIFY      VERIFY      VERIFY
                     ↓           ↓           ↓
                BENCHMARK   BENCHMARK   BENCHMARK
                     └───────────┼───────────┘
                                 ↓
                            🏆 RANKING
                                 ↓
                       🧠 RECOMMENDATION
                                 ↓
                    ┌────────────┴────────────┐
                    ↓                         ↓
              Best Candidate            Best Language
                    │                         │
                    └────────────┬────────────┘
                                 ↓
                           FINAL REPORT
'''                           

'\n🚀 The entire OptiForge intelligence pipeline\n\n                         ┌───────────────┐\n                         │ SOURCE CODE   │\n                         └───────┬───────┘\n                                 ↓\n                         ┌───────────────┐\n                         │   ANALYZER    │\n                         └───────┬───────┘\n                                 ↓\n                         ┌───────────────┐\n                         │ LLM GENERATOR │\n                         └───────┬───────┘\n                                 ↓\n                     ┌───────────┼───────────┐\n                     ↓           ↓           ↓\n                    C1          C2          C3 ...\n                     ↓           ↓           ↓\n                 COMPILE     COMPILE     COMPILE\n                     ↓           ↓           ↓\n                  VERIFY      VERIFY      VERIFY\n                     ↓           ↓           ↓\n                BENCHMARK   BENCHMARK   BENCHMARK\n  

UI Phase begins 😁

# Phase 10 — OptiForge AI User Interface

In this phase, we connect the OptiForge engine to a Gradio interface.

The UI will allow the user to:

- Enter source code
- Select source language
- Select target language
- Choose an optimization goal
- Choose a model strategy
- Provide test cases
- Run OptiForge
- View analysis
- View generated candidates
- View verification and benchmark results
- View the final recommendation

The interface is only the presentation layer.

The actual decisions are still made by the OptiForge engine.

## 🔎 Phase 10.1 — Check Engine Interfaces

Before building the UI, verify the interfaces of the optimization engine.

The UI will act only as a presentation layer.

In [205]:
import inspect

print("evaluate_candidate:")
print(inspect.signature(evaluate_candidate))

print("\nrank_candidates:")
print(inspect.signature(rank_candidates))

print("\nbuild_recommendation:")
print(inspect.signature(build_recommendation))

print("\ngenerate_candidates:")
print(inspect.signature(generate_candidates))

evaluate_candidate:
(candidate, original_code, original_language, test_inputs, benchmark_input)

rank_candidates:
(candidates)

build_recommendation:
(candidates)

generate_candidates:
(source_code, source_language, target_language, profile, models)


## 🎛️ Phase 10.2 — OptiForge UI Controller

The Gradio interface should not contain optimization logic.

The controller connects the UI to the existing OptiForge engine:

Source Code
→ Analysis
→ AI Transformation
→ Candidate Evaluation
→ Verification
→ Benchmark
→ Recommendation

In [206]:
def optimize_ui(source_code, source_language, target_language, model):
    if not source_code.strip():
        return (
            "",
            "❌ Please enter source code.",
            "❌ No code provided."
        )

    try:
        # Analyze source code
        profile = analyze_code(source_code, source_language)

        # Generate candidate
        candidates = generate_candidates(
            source_code=source_code,
            source_language=source_language,
            target_language=target_language,
            profile=profile,
            models=[model],
            optimization_goal="Balanced"
        )

        # Evaluate candidate
        for candidate in candidates:
            evaluate_candidate(
                candidate,
                source_code,
                source_language,
                [""],
                ""
            )

        # Rank candidates
        ranked_candidates = rank_candidates(candidates)

        # Build recommendation
        recommendation = build_recommendation(ranked_candidates)

        # Select best candidate
        best = recommendation.recommended_candidate

        if best is None:
            return (
                "",
                "❌ No valid optimized candidate was produced.",
                recommendation.reason
            )

        generated_code = best.source_code

        result_text = (
            f"Compilation: {'PASS' if best.compiled else 'FAIL'}\n"
            f"Correctness: {'PASS' if best.verified else 'FAIL'}\n"
        )

        if best.benchmark:
            result_text += (
                f"\nOriginal Runtime: "
                f"{best.benchmark.original_average:.6f} s"
                f"\nOptimized Runtime: "
                f"{best.benchmark.generated_average:.6f} s"
                f"\nSpeedup: "
                f"{best.benchmark.speedup:.2f}x"
                f"\nImprovement: "
                f"{best.benchmark.improvement_percent:.2f}%"
            )

        return (
            generated_code,
            result_text,
            recommendation.reason
        )

    except Exception as e:
        return (
            "",
            f"❌ Optimization failed:\n{str(e)}",
            "Please check the source code, selected languages, and model."
        )

## 🤖 Phase 10.3 — OptiForge Model Selection

The user can choose the LLM used for code transformation.

The UI displays friendly model names, while OptiForge internally
uses the corresponding model IDs.

In [210]:
MODEL_OPTIONS = {
    "Qwen 3.5 2B (Local)": "qwen3.5:2b",
    "Llama 3.2 (Local)": "llama3.2:latest",
    "GPT-OSS 120B (Groq)": "openai/gpt-oss-120b",
    "Gemini 2.5 Pro": "gemini-2.5-pro",
    "Qwen3 Coder 30B (OpenRouter)": "qwen/qwen3-coder-30b-a3b-instruct"
}

print("Available models:")

for name in MODEL_OPTIONS:
    print("-", name)

Available models:
- Qwen 3.5 2B (Local)
- Llama 3.2 (Local)
- GPT-OSS 120B (Groq)
- Gemini 2.5 Pro
- Qwen3 Coder 30B (OpenRouter)


In [213]:
def get_selected_model(model_display_name):
    if model_display_name not in MODEL_OPTIONS:
        raise ValueError(f"Unknown model: {model_display_name}")

    return MODEL_OPTIONS[model_display_name]


selected_model = get_selected_model(
    "Qwen3 Coder 30B (OpenRouter)"
)

print("Selected model:")
print(selected_model)

Selected model:
qwen/qwen3-coder-30b-a3b-instruct


## ⚙️ Phase 10.4 — Model-Aware Optimization Controller

The user selects the LLM from the interface.

The selected model is passed to the existing candidate-generation engine.

The optimization strategy remains "Balanced" conceptually,
but it is not exposed as a model-selection mechanism.

In [218]:
def optimize_ui(
    source_code,
    source_language,
    target_language,
    model_display_name
):
    if not source_code.strip():
        return "", "❌ Please enter source code.", ""

    try:
        # 1. Get the actual model ID selected by the user
        selected_model = get_selected_model(model_display_name)

        # 2. Analyze source code
        profile = analyze_code(
            source_code,
            source_language
        )

        # 3. Generate candidate using selected model
        candidates = generate_candidates(
            source_code=source_code,
            source_language=source_language,
            target_language=target_language,
            profile=profile,
            models=[selected_model]
        )

        # 4. Evaluate candidate
        for candidate in candidates:
            evaluate_candidate(
                candidate,
                source_code,
                source_language,
                [""],
                ""
            )

        # 5. Rank candidates
        ranked_candidates = rank_candidates(candidates)

        # 6. Build recommendation
        recommendation = build_recommendation(
            ranked_candidates
        )

        # 7. No valid candidate
        if recommendation.recommended_candidate is None:
            return (
                "",
                "❌ No valid optimized candidate.",
                recommendation.reason
            )

        best = recommendation.recommended_candidate

        # 8. Build clean result
        result = (
            f"### Compilation\n"
            f"{'✅ PASS' if best.compiled else '❌ FAIL'}\n\n"
            f"### Correctness\n"
            f"{'✅ PASS' if best.verified else '❌ FAIL'}\n"
        )

        if best.benchmark:
            benchmark = best.benchmark

            result += (
                f"\n### Performance\n"
                f"- Original: `{benchmark.original_average:.6f} s`\n"
                f"- Optimized: `{benchmark.generated_average:.6f} s`\n"
                f"- Speedup: `{benchmark.speedup:.2f}×`\n"
                f"- Improvement: `{benchmark.improvement_percent:.2f}%`\n"
            )

        return (
            best.source_code,
            result,
            recommendation.reason
        )

    except Exception as e:
        return (
            "",
            f"❌ Optimization failed:\n\n`{str(e)}`",
            f"Selected model: {model_display_name}"
        )

In [219]:
test_code = """
print(sum(range(100000)))
"""

generated, result, recommendation = optimize_ui(
    test_code,
    "python",
    "cpp",
    "Qwen3 Coder 30B (OpenRouter)"
)

print("GENERATED CODE:")
print(generated)

print("\nRESULT:")
print(result)

print("\nRECOMMENDATION:")
print(recommendation)

GENERATED CODE:
#include <iostream>

int main() {
    std::cout << 4999950000LL << std::endl;
    return 0;
}

RESULT:
### Compilation
✅ PASS

### Correctness
✅ PASS

### Performance
- Original: `0.176672 s`
- Optimized: `0.139757 s`
- Speedup: `1.26×`
- Improvement: `20.89%`


RECOMMENDATION:
The selected candidate passed verification and achieved the best measured runtime.


# 🎨 Phase 10.5 — OptiForge Interface

OptiForge provides a minimal interface for transforming and
optimizing source code.

The user selects:

- Source language
- Target language
- LLM model

Then OptiForge generates, verifies, benchmarks, and evaluates
the transformed program.

In [220]:
import gradio as gr

## 📊 Phase 10.5 — Result State & Comparison Store

OptiForge keeps benchmark results for the current source program.

Changing the target language or model adds/updates a result.

Changing the source code creates a new analysis session and clears
the previous comparison results.

In [324]:
import hashlib

optiforge_state = {
    "source_hash": None,
    "results": {}
}


def get_source_hash(source_code, source_language):
    source_identity = f"{source_language}\n{source_code}"

    return hashlib.sha256(
        source_identity.encode("utf-8")
    ).hexdigest()
    

In [325]:
def update_source_state(source_code, source_language):
    new_hash = get_source_hash(
        source_code,
        source_language
    )

    if optiforge_state["source_hash"] != new_hash:
        optiforge_state["source_hash"] = new_hash
        optiforge_state["results"] = {}
        optiforge_state["baseline"] = None
        optiforge_state["analysis"] = None

    return optiforge_state["source_hash"]

In [327]:
code_a = "print(10)"
code_b = "print(20)"

update_source_state(code_a,"python")

print(optiforge_state)

{'source_hash': '05bdb696e1bec8a393999ae504319792f20cde6185896879d39541580f192704', 'results': {}, 'baseline': None, 'analysis': None}


In [251]:
optiforge_state["results"]["cpp"] = {
    "runtime": 0.12
}

print(optiforge_state["results"])

update_source_state(code_a)

print("Same code:")
print(optiforge_state["results"])

update_source_state(code_b)

print("Changed code:")
print(optiforge_state["results"])

{'cpp': {'runtime': 0.12}}
Same code:
{'cpp': {'runtime': 0.12}}
Changed code:
{}


## 10.5.3 — Optimization Result Records

Each successful optimization is stored against:

- Target language
- Selected model

This allows OptiForge to preserve previous results when the user
changes only the target language or model.

When the source code changes, the complete result state is reset.

In [252]:
def make_result_key(target_language, model):
    return f"{target_language}::{model}"


def store_optimization_result(
    target_language,
    model,
    generated_code,
    candidate,
    recommendation
):
    key = make_result_key(target_language, model)

    optiforge_state["results"][key] = {
        "target_language": target_language,
        "model": model,
        "generated_code": generated_code,
        "candidate": candidate,
        "recommendation": recommendation
    }

    return key

In [328]:
update_source_state(code_a, "python")
python_hash = optiforge_state["source_hash"]

update_source_state(code_a, "java")
java_hash = optiforge_state["source_hash"]

print("Python hash:", python_hash)
print("Java hash:  ", java_hash)
print("Different:", python_hash != java_hash)

Python hash: 05bdb696e1bec8a393999ae504319792f20cde6185896879d39541580f192704
Java hash:   a0179f47347ee71d48070a76644c142c3c71cf330d7961272deaa47f4f89f3da
Different: True


In [329]:
import inspect

print(inspect.signature(get_source_hash))
print(inspect.signature(update_source_state))

(source_code, source_language)
(source_code, source_language)


In [330]:
store_optimization_result(
    target_language="java",
    model="Qwen3 Coder 30B (OpenRouter)",
    generated_code="public class Main {...}",
    candidate=None,
    recommendation="Java recommendation"
)

'java::Qwen3 Coder 30B (OpenRouter)'

In [331]:
print(optiforge_state["results"].keys())

dict_keys(['java::Qwen3 Coder 30B (OpenRouter)'])


## 10.5.5 — Inspect the Optimization Controller

Before connecting the result store, inspect the current `optimize_ui`
function so we can integrate state management without breaking the
working optimization pipeline.

In [332]:
import inspect

print("OPTIMIZE_UI SIGNATURE:")
print(inspect.signature(optimize_ui))

print("\nOPTIMIZE_UI SOURCE:")
print(inspect.getsource(optimize_ui))

OPTIMIZE_UI SIGNATURE:
(source_code, source_language, target_language, model_display_name)

OPTIMIZE_UI SOURCE:
def optimize_ui(
    source_code,
    source_language,
    target_language,
    model_display_name
):
    if not source_code.strip():
        return "", "❌ Please enter source code.", "", ""

    try:
        # 1. Detect source changes
        update_source_state(
            source_code,
            source_language
        )

        # 2. Resolve selected model
        selected_model = get_selected_model(model_display_name)

        # 3. Create/reuse baseline
        store_baseline(
            source_code,
            source_language
        )

        # 4. Create/reuse algorithm analysis
        store_algorithm_analysis(
            source_code,
            source_language,
            selected_model
        )

        # 5. Check transformation cache
        result_key = make_result_key(
            target_language,
            model_display_name
        )

        existin

## 10.5.6 — State-Aware Optimization Controller

The controller now remembers results for the current source program.

- Same source + new target → keep previous results.
- Same source + new model → keep previous results.
- Changed source → clear all previous results.
- The selected model is always used explicitly.

In [372]:
def optimize_ui(
    source_code,
    source_language,
    target_language,
    model_display_name
):
    if not source_code.strip():
        return "", "", "", "", "", ""

    try:
        # 1. Detect source changes
        update_source_state(
            source_code,
            source_language
        )

        # 2. Resolve selected model
        selected_model = get_selected_model(
            model_display_name
        )

        # 3. Get/reuse source analysis
        analysis_bundle = get_source_analysis_bundle(
            source_code,
            source_language,
            selected_model
        )

        profile = analysis_bundle["profile"]

        # 4. Create/reuse baseline
        store_baseline(
            source_code,
            source_language
        )

        # 5. Check transformation cache
        result_key = make_result_key(
            target_language,
            model_display_name
        )

        existing_result = optiforge_state["results"].get(
            result_key
        )

        # 6. If already evaluated, reuse it
        if existing_result is not None:

            return (
                existing_result["generated_code"],
                render_language_comparison(),
                render_algorithm_analysis(),
                render_code_insights(profile),
                render_performance_hotspots(profile),
                existing_result["recommendation"].reason
            )

        # 7. Generate candidate
        candidates = generate_candidates(
            source_code=source_code,
            source_language=source_language,
            target_language=target_language,
            profile=profile,
            models=[selected_model]
        )

        # 8. Evaluate candidate
        for candidate in candidates:
            evaluate_candidate(
                candidate,
                source_code,
                source_language,
                [""],
                ""
            )

        # 9. Rank candidates
        ranked_candidates = rank_candidates(
            candidates
        )

        # 10. Build recommendation
        recommendation = build_recommendation(
            ranked_candidates
        )

        # 11. No valid candidate
        if recommendation.recommended_candidate is None:

            return (
                "",
                render_language_comparison(),
                render_algorithm_analysis(),
                render_code_insights(profile),
                render_performance_hotspots(profile),
                f"❌ {recommendation.reason}"
            )

        # 12. Select best candidate
        best = recommendation.recommended_candidate

        # 13. Store result
        store_optimization_result(
            target_language=target_language,
            model=model_display_name,
            generated_code=best.source_code,
            candidate=best,
            recommendation=recommendation
        )

        # 14. Return complete result
        return (
            best.source_code,
            render_language_comparison(),
            render_algorithm_analysis(),
            render_code_insights(profile),
            render_performance_hotspots(profile),
            recommendation.reason
        )

    except Exception as e:

        return (
            "",
            render_language_comparison(),
            render_algorithm_analysis(),
            "### 🔍 Code Insights\n\nAnalysis unavailable.",
            "### 🔥 Performance Hotspots\n\nAnalysis unavailable.",
            f"❌ Optimization failed: `{str(e)}`"
        )

In [374]:
test_code = """
n = 1000000
total = 0

for i in range(1, n + 1):
    total += i

print(total)
"""

In [375]:
print(type(test_code))
print(test_code)

<class 'str'>

n = 1000000
total = 0

for i in range(1, n + 1):
    total += i

print(total)



In [376]:
generated, comparison, algorithm, insights, hotspots, recommendation = optimize_ui(
    test_code,
    "python",
    "cpp",
    "Qwen3 Coder 30B (OpenRouter)"
)

print("=== COMPARISON ===")
print(comparison)

print("\n=== ALGORITHM ===")
print(algorithm)

print("\n=== INSIGHTS ===")
print(insights)

print("\n=== HOTSPOTS ===")
print(hotspots)

print("\n=== RECOMMENDATION ===")
print(recommendation)

=== COMPARISON ===

### 🌍 Language Comparison

| Language | Runtime | Speedup | Improvement | Correctness | Status |
|---|---:|---:|---:|---|---|
| Python | 0.245608 | 1.00× | — | BASELINE | Baseline |
| Cpp | 0.148601 | 2.28× | 56.23% | PASS | Tested |
| Java | Not tested | — | — | — | Not tested |


=== ALGORITHM ===

### 🧠 Algorithm Analysis

| Metric | Analysis |
|---|---|
| **Algorithm** | Summation |
| **Approach** | Iterative summation of consecutive integers from 1 to n |
| **Time Complexity** | `O(n)` |
| **Space Complexity** | `O(1)` |
| **Optimization Opportunity** | Can be optimized to O(1) using the mathematical formula n*(n+1)/2 |


=== INSIGHTS ===
### 🔍 Code Insights

• 🔁 1 loop(s) detected
• 🧩 No user-defined functions detected
• 📦 No external imports detected
• 💻 I/O operations detected: output
• ⏱️ Complexity hints: Possible linear iteration: O(n)

=== HOTSPOTS ===
### 🔥 Performance Hotspots

• ⏱️ Possible linear iteration: O(n)
• 🔁 1 loop(s) may contribute to runtim

In [377]:
print(inspect.signature(optimize_ui))
print(optiforge_state)

(source_code, source_language, target_language, model_display_name)
{'source_hash': '5994a3a5af2156540ec9ca8cf7d2c51adb621da72cc839113c69e2926bb2d1e8', 'results': {'cpp::Qwen3 Coder 30B (OpenRouter)': {'target_language': 'cpp', 'model': 'Qwen3 Coder 30B (OpenRouter)', 'generated_code': '#include <iostream>\n\nint main() {\n    const long long n = 1000000;\n    long long total = 0;\n    \n    for (long long i = 1; i <= n; ++i) {\n        total += i;\n    }\n    \n    std::cout << total << std::endl;\n    return 0;\n}', 'candidate': CandidateResult(candidate_id='C1', model='qwen/qwen3-coder-30b-a3b-instruct', source_code='#include <iostream>\n\nint main() {\n    const long long n = 1000000;\n    long long total = 0;\n    \n    for (long long i = 1; i <= n; ++i) {\n        total += i;\n    }\n    \n    std::cout << total << std::endl;\n    return 0;\n}', language='cpp', compiled=True, verified=True, benchmark=BenchmarkResult(original_runs=[0.33839110005646944, 0.33510290004778653, 0.33544

In [337]:
def build_language_comparison():
    """
    Build a Python / C++ / Java comparison
    using the original source as the baseline.
    """

    baseline = optiforge_state.get("baseline")

    rows = []

    for language in ["python", "cpp", "java"]:

        # Original source language
        if baseline and language == baseline["language"]:
            rows.append({
                "Language": language.capitalize(),
                "Runtime (s)": f"{baseline['runtime']:.6f}",
                "Speedup": "1.00×",
                "Improvement": "—",
                "Correctness": "BASELINE",
                "Status": "Baseline"
            })
            continue

        # Look for a generated result
        matching_results = [
            result
            for result in optiforge_state["results"].values()
            if result["target_language"] == language
        ]

        if not matching_results:
            rows.append({
                "Language": language.capitalize(),
                "Runtime (s)": "Not tested",
                "Speedup": "—",
                "Improvement": "—",
                "Correctness": "—",
                "Status": "Not tested"
            })
            continue

        stored = matching_results[-1]
        candidate = stored["candidate"]
        benchmark = candidate.benchmark

        rows.append({
            "Language": language.capitalize(),
            "Runtime (s)": f"{benchmark.generated_average:.6f}",
            "Speedup": f"{benchmark.speedup:.2f}×",
            "Improvement": f"{benchmark.improvement_percent:.2f}%",
            "Correctness": "PASS" if candidate.verified else "FAIL",
            "Status": "Tested"
        })

    return rows

In [338]:
comparison = build_language_comparison()

for row in comparison:
    print(row)

{'Language': 'Python', 'Runtime (s)': '0.244273', 'Speedup': '1.00×', 'Improvement': '—', 'Correctness': 'BASELINE', 'Status': 'Baseline'}
{'Language': 'Cpp', 'Runtime (s)': 'Not tested', 'Speedup': '—', 'Improvement': '—', 'Correctness': '—', 'Status': 'Not tested'}
{'Language': 'Java', 'Runtime (s)': 'Not tested', 'Speedup': '—', 'Improvement': '—', 'Correctness': '—', 'Status': 'Not tested'}


In [339]:
optiforge_state["baseline"] = None

In [340]:
def store_baseline(source_code, source_language):
    """
    Benchmark and store the original source program.
    """

    if optiforge_state["baseline"] is not None:
        return optiforge_state["baseline"]

    benchmark = benchmark_program(
        source_code,
        source_language,
        input_data="",
        runs=5
    )

    optiforge_state["baseline"] = {
        "language": source_language,
        "runtime": benchmark.original_average
    }

    return optiforge_state["baseline"]

In [341]:
test_benchmark = benchmark_program(
    test_code,
    "python",
    input_data="",
    runs=5
)

print("Type:", type(test_benchmark))
print("Value:", test_benchmark)

Type: <class 'tuple'>
Value: ([0.26114189997315407, 0.24832639994565398, 0.23158490005880594, 0.28182860009837896, 0.22220620000734925], None)


In [342]:
def store_baseline(source_code, source_language):
    """
    Benchmark and store the original source program.
    """

    if optiforge_state.get("baseline") is not None:
        return optiforge_state["baseline"]

    runtimes, error = benchmark_program(
        source_code,
        source_language,
        input_data="",
        runs=5
    )

    if error is not None:
        raise RuntimeError(f"Baseline benchmark failed: {error}")

    average_runtime = sum(runtimes) / len(runtimes)

    optiforge_state["baseline"] = {
        "language": source_language,
        "runtime": average_runtime,
        "runs": runtimes
    }

    return optiforge_state["baseline"]

In [343]:
baseline = store_baseline(
    test_code,
    "python"
)

print(baseline)

{'language': 'python', 'runtime': 0.23907153999898584, 'runs': [0.22899380000308156, 0.24580119992606342, 0.25047170009929687, 0.23931350000202656, 0.23077749996446073]}


In [344]:
def render_language_comparison():
    rows = build_language_comparison()

    markdown = """
### 🌍 Language Comparison

| Language | Runtime | Speedup | Improvement | Correctness | Status |
|---|---:|---:|---:|---|---|
"""

    for row in rows:
        markdown += (
            f"| {row['Language']} "
            f"| {row['Runtime (s)']} "
            f"| {row['Speedup']} "
            f"| {row['Improvement']} "
            f"| {row['Correctness']} "
            f"| {row['Status']} |\n"
        )

    return markdown

In [345]:
print(render_language_comparison())


### 🌍 Language Comparison

| Language | Runtime | Speedup | Improvement | Correctness | Status |
|---|---:|---:|---:|---|---|
| Python | 0.239072 | 1.00× | — | BASELINE | Baseline |
| Cpp | Not tested | — | — | — | Not tested |
| Java | Not tested | — | — | — | Not tested |



In [409]:
OPTIFORGE_CSS = """
#app-container {
    max-width: 1400px;
    margin: auto;
}

#title {
    text-align: center;
    margin-bottom: 25px;
}

.editor-column {
    min-width: 0;
}

#optimize-button {
    max-width: 240px;
    margin: 20px auto;
}

#results {
    margin-top: 20px;
}

.footer {
    text-align: center;
    opacity: 0.65;
    margin-top: 30px;
}
"""


with gr.Blocks(
    title="OptiForge",
    css=OPTIFORGE_CSS
) as app:

    # Header
    gr.Markdown(
        "# OptiForge.Ai",
        elem_id="title"
    )

    # Source / Generated code
    with gr.Row():

        with gr.Column(
            scale=1,
            elem_classes="editor-column"
        ):
            source_code = gr.Code(
                label="SOURCE CODE",
                lines=24
            )

        with gr.Column(
            scale=1,
            elem_classes="editor-column"
        ):
            generated_code = gr.Code(
                label="GENERATED CODE",
                lines=24,
                interactive=False
            )

    # Controls
    with gr.Row():

        source_language = gr.Dropdown(
            choices=["python", "cpp", "java"],
            value="python",
            label="Source Language"
        )

        target_language = gr.Dropdown(
            choices=["python", "cpp", "java"],
            value="cpp",
            label="Target Language"
        )

        model = gr.Dropdown(
            choices=list(MODEL_OPTIONS.keys()),
            value="Qwen3 Coder 30B (OpenRouter)",
            label="Model"
        )

    # Optimize button
    optimize_button = gr.Button(
        "Optimize Code",
        variant="primary",
        elem_id="optimize-button"
    )

    # Results
    gr.Markdown("## Results")

    comparison_result = gr.Markdown(
        "Run an optimization to see language comparison.",
        elem_id="results"
    )

    algorithm_result = gr.Markdown(
        "Algorithm analysis will appear here."
    )

    insights_result = gr.Markdown(
        "Code insights will appear here."
    )

    hotspots_result = gr.Markdown(
        "Performance hotspots will appear here."
    )

    recommendation = gr.Markdown()

    verification_result = gr.Markdown(
        "Verification results will appear here."
    )

    # Footer
    gr.Markdown(
        "<div class='footer'>Built with Gradio</div>"
    )

    # Button event
    optimize_button.click(
        fn=optimize_ui_with_verification,
        inputs=[
            source_code,
            source_language,
            target_language,
            model
        ],
        outputs=[
            generated_code,
            comparison_result,
            algorithm_result,
            verification_result,
            insights_result,
            hotspots_result,
            recommendation
        ]
    )

In [347]:
print("DSAAnalysis:", "DSAAnalysis" in globals())
print("build_recommendation_prompt:", "build_recommendation_prompt" in globals())
print("generate_recommendation_explanation:", "generate_recommendation_explanation" in globals())
print("display_recommendation:", "display_recommendation" in globals())

DSAAnalysis: True
build_recommendation_prompt: True
generate_recommendation_explanation: True
display_recommendation: True


In [348]:
import inspect

print("DSAAnalysis:")
print(inspect.signature(DSAAnalysis))

print("\nbuild_recommendation_prompt:")
print(inspect.signature(build_recommendation_prompt))

print("\ngenerate_recommendation_explanation:")
print(inspect.signature(generate_recommendation_explanation))

print("\ndisplay_recommendation:")
print(inspect.signature(display_recommendation))

DSAAnalysis:
(algorithm: str = '', time_complexity: str = '', space_complexity: str = '', approach: str = '', optimization_opportunity: str = '') -> None

build_recommendation_prompt:
(original_language, candidates, dsa_analysis=None)

generate_recommendation_explanation:
(model, original_language, candidates, dsa_analysis=None)

display_recommendation:
(result)


In [349]:
import json
import re

ALGORITHM_ANALYSIS_SYSTEM_PROMPT = """
You are the algorithm analysis engine of OptiForge AI.

Analyze the provided source code and identify its algorithmic characteristics.

Return ONLY valid JSON with exactly these fields:

{
    "algorithm": "...",
    "approach": "...",
    "time_complexity": "...",
    "space_complexity": "...",
    "optimization_opportunity": "..."
}

Rules:
- Identify the actual algorithm or computational approach used.
- Give Big-O time complexity.
- Give Big-O auxiliary space complexity.
- Do not invent algorithms that are not present.
- Keep explanations concise and technically accurate.
- If the code is not a traditional DSA algorithm, describe the main computational approach.
- Optimization opportunities must be realistic.
"""

In [350]:
def analyze_algorithm(
    model,
    source_code,
    source_language,
    profile=None
):
    client = clients[model]

    profile_text = ""

    if profile is not None:
        profile_text = f"""
Static analysis information:

- Total lines: {profile.total_lines}
- Code lines: {profile.code_lines}
- Comment lines: {profile.comment_lines}
- Blank lines: {profile.blank_lines}
- Functions: {profile.functions}
- Loops: {profile.loops}
- Imports: {profile.imports}
- I/O operations: {profile.io_operations}
- Complexity hints: {profile.complexity_hints}
- Performance patterns: {profile.performance_patterns}
"""

    user_prompt = f"""
Analyze this {source_language} program.

{profile_text}

SOURCE CODE:
{source_code}
"""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": ALGORITHM_ANALYSIS_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        temperature=0
    )

    content = response.choices[0].message.content.strip()

    # Remove possible markdown JSON fences
    content = re.sub(
        r"^```(?:json)?\s*|\s*```$",
        "",
        content,
        flags=re.IGNORECASE
    ).strip()

    data = json.loads(content)

    return DSAAnalysis(
        algorithm=data.get("algorithm", ""),
        approach=data.get("approach", ""),
        time_complexity=data.get("time_complexity", ""),
        space_complexity=data.get("space_complexity", ""),
        optimization_opportunity=data.get(
            "optimization_opportunity",
            ""
        )
    )

In [351]:
profile = analyze_code(
    test_code,
    "python"
)

dsa_analysis = analyze_algorithm(
    model="qwen/qwen3-coder-30b-a3b-instruct",
    source_code=test_code,
    source_language="python",
    profile=profile
)

print("Algorithm:", dsa_analysis.algorithm)
print("Approach:", dsa_analysis.approach)
print("Time Complexity:", dsa_analysis.time_complexity)
print("Space Complexity:", dsa_analysis.space_complexity)
print("Optimization Opportunity:", dsa_analysis.optimization_opportunity)

Algorithm: Summation
Approach: Iterative summation using a for loop to calculate the sum of integers from 1 to n
Time Complexity: O(n)
Space Complexity: O(1)
Optimization Opportunity: Can be optimized to O(1) using the mathematical formula for sum of first n natural numbers: n*(n+1)/2


In [352]:
optiforge_state["analysis"] = None

print(optiforge_state)

{'source_hash': '7364e14297d59d0453f55807b5e1e3fc4e65e5041222972970fecffdff5ab7d6', 'results': {}, 'baseline': {'language': 'python', 'runtime': 0.23907153999898584, 'runs': [0.22899380000308156, 0.24580119992606342, 0.25047170009929687, 0.23931350000202656, 0.23077749996446073]}, 'analysis': None}


In [353]:
def store_algorithm_analysis(
    source_code,
    source_language,
    model
):
    """
    Analyze the source code once and keep the
    analysis while the source remains unchanged.
    """

    # Return cached analysis if available
    if optiforge_state.get("analysis") is not None:
        return optiforge_state["analysis"]

    # Analyze source
    profile = analyze_code(
        source_code,
        source_language
    )

    analysis = analyze_algorithm(
        model=model,
        source_code=source_code,
        source_language=source_language,
        profile=profile
    )

    optiforge_state["analysis"] = analysis

    return analysis

In [354]:
optiforge_state["baseline"] = None

In [355]:
analysis = store_algorithm_analysis(
    test_code,
    "python",
    "qwen/qwen3-coder-30b-a3b-instruct"
)

print("Algorithm:", analysis.algorithm)
print("Approach:", analysis.approach)
print("Time:", analysis.time_complexity)
print("Space:", analysis.space_complexity)
print("Optimization:", analysis.optimization_opportunity)

Algorithm: Summation
Approach: Iterative summation of consecutive integers from 1 to n
Time: O(n)
Space: O(1)
Optimization: Can be optimized to O(1) using the mathematical formula n*(n+1)/2


In [356]:
def render_algorithm_analysis():
    analysis = optiforge_state.get("analysis")

    if analysis is None:
        return "### 🧠 Algorithm Analysis\n\nNot analyzed yet."

    return f"""
### 🧠 Algorithm Analysis

| Metric | Analysis |
|---|---|
| **Algorithm** | {analysis.algorithm} |
| **Approach** | {analysis.approach} |
| **Time Complexity** | `{analysis.time_complexity}` |
| **Space Complexity** | `{analysis.space_complexity}` |
| **Optimization Opportunity** | {analysis.optimization_opportunity} |
"""

###  Code Insights & Performance Hotspots

In [360]:
'''
Source Code
    ↓
CodeProfile
    ↓
┌─────────────────────────┐
│ 🔍 Code Insights         │
│ • loops                 │
│ • functions             │
│ • dependencies          │
│ • I/O                   │
│ • complexity hints      │
│ • performance patterns  │
└─────────────────────────┘
'''

'\nSource Code\n    ↓\nCodeProfile\n    ↓\n┌─────────────────────────┐\n│ 🔍 Code Insights         │\n│ • loops                 │\n│ • functions             │\n│ • dependencies          │\n│ • I/O                   │\n│ • complexity hints      │\n│ • performance patterns  │\n└─────────────────────────┘\n'

In [381]:
def render_code_insights(profile=None):
    if profile is None:
        return "### 🔍 Code Insights\n\nNot analyzed yet."

    insights = []

    # Loop information
    if profile.loops > 0:
        insights.append(
            f"-> 🔁 {profile.loops} loop(s) detected"
        )
    else:
        insights.append(
            "• 🔁 No loops detected"
        )

    # Function information
    if profile.functions:
        insights.append(
            f"-> 🧩 {len(profile.functions)} function(s) detected"
        )
    else:
        insights.append(
            "• 🧩 No user-defined functions detected"
        )

    # Imports / dependencies
    if profile.imports:
        insights.append(
            f"-> 📦 {len(profile.imports)} import(s)/dependency(ies) detected"
        )
    else:
        insights.append(
            "• 📦 No external imports detected"
        )

    # I/O
    if profile.io_operations:
        insights.append(
            f"-> 💻 I/O operations detected: "
            f"{', '.join(profile.io_operations)}"
        )
    else:
        insights.append(
            "• 💻 No explicit I/O operations detected"
        )

    # Complexity hints
    if profile.complexity_hints:
        insights.append(
            "-> ⏱️ Complexity hints: "
            + "; ".join(profile.complexity_hints)
        )

    # Performance patterns
    if profile.performance_patterns:
        insights.append(
            "-> 🔥 Performance patterns: "
            + "; ".join(profile.performance_patterns)
        )

    return (
        "### 🔍 Code Insights\n\n"
        + "\n".join(insights)
    )

In [382]:
profile = analyze_code(
    test_code,
    "python"
)

print(render_code_insights(profile))

### 🔍 Code Insights

-> 🔁 1 loop(s) detected
• 🧩 No user-defined functions detected
• 📦 No external imports detected
-> 💻 I/O operations detected: output
-> ⏱️ Complexity hints: Possible linear iteration: O(n)


In [363]:
def render_performance_hotspots(profile=None):
    if profile is None:
        return "### 🔥 Performance Hotspots\n\nNot analyzed yet."

    hotspots = []

    # Complexity-based hotspot
    if profile.complexity_hints:
        for hint in profile.complexity_hints:
            hotspots.append(f"• ⏱️ {hint}")

    # Performance patterns
    if profile.performance_patterns:
        for pattern in profile.performance_patterns:
            hotspots.append(f"• 🔥 {pattern}")

    # Loop-based hotspot
    if profile.loops > 0:
        hotspots.append(
            f"• 🔁 {profile.loops} loop(s) may contribute to runtime"
        )

    if not hotspots:
        hotspots.append(
            "• No obvious performance hotspots detected"
        )

    return (
        "### 🔥 Performance Hotspots\n\n"
        + "\n".join(hotspots)
    )

In [364]:
profile = analyze_code(
    test_code,
    "python"
)

print(render_performance_hotspots(profile))

### 🔥 Performance Hotspots

• ⏱️ Possible linear iteration: O(n)
• 🔁 1 loop(s) may contribute to runtime


In [365]:
optiforge_state["profile"] = None

print(optiforge_state.keys())

dict_keys(['source_hash', 'results', 'baseline', 'analysis', 'profile'])


In [366]:
def store_code_profile(source_code, source_language):
    """
    Analyze and cache the source code profile.
    """

    if optiforge_state.get("profile") is not None:
        return optiforge_state["profile"]

    profile = analyze_code(
        source_code,
        source_language
    )

    optiforge_state["profile"] = profile

    return profile

In [367]:
def update_source_state(source_code, source_language):
    new_hash = get_source_hash(
        source_code,
        source_language
    )

    if optiforge_state["source_hash"] != new_hash:
        optiforge_state["source_hash"] = new_hash
        optiforge_state["results"] = {}
        optiforge_state["baseline"] = None
        optiforge_state["analysis"] = None
        optiforge_state["profile"] = None

    return optiforge_state["source_hash"]

In [368]:
profile = store_code_profile(
    test_code,
    "python"
)

print("Language:", profile.language)
print("Total lines:", profile.total_lines)
print("Code lines:", profile.code_lines)
print("Loops:", profile.loops)
print("Functions:", profile.functions)
print("Imports:", profile.imports)
print("I/O:", profile.io_operations)
print("Complexity hints:", profile.complexity_hints)
print("Performance patterns:", profile.performance_patterns)

Language: python
Total lines: 9
Code lines: 5
Loops: 1
Functions: []
Imports: []
I/O: ['output']
Complexity hints: ['Possible linear iteration: O(n)']
Performance patterns: []


In [369]:
print(render_code_insights(profile))
print()
print(render_performance_hotspots(profile))

### 🔍 Code Insights

• 🔁 1 loop(s) detected
• 🧩 No user-defined functions detected
• 📦 No external imports detected
• 💻 I/O operations detected: output
• ⏱️ Complexity hints: Possible linear iteration: O(n)

### 🔥 Performance Hotspots

• ⏱️ Possible linear iteration: O(n)
• 🔁 1 loop(s) may contribute to runtime


In [370]:
def get_source_analysis_bundle(
    source_code,
    source_language,
    model
):
    """
    Get or create all source-level analysis.
    """

    profile = store_code_profile(
        source_code,
        source_language
    )

    algorithm = store_algorithm_analysis(
        source_code,
        source_language,
        model
    )

    return {
        "profile": profile,
        "algorithm": algorithm
    }

In [371]:
analysis_bundle = get_source_analysis_bundle(
    test_code,
    "python",
    "qwen/qwen3-coder-30b-a3b-instruct"
)

print("Profile:", type(analysis_bundle["profile"]))
print("Algorithm:", type(analysis_bundle["algorithm"]))

print("\nAlgorithm:", analysis_bundle["algorithm"].algorithm)
print("Time:", analysis_bundle["algorithm"].time_complexity)

Profile: <class '__main__.CodeProfile'>
Algorithm: <class '__main__.DSAAnalysis'>

Algorithm: Linear Search
Time: O(n)


In [383]:
import inspect

print("verify_test_case:", inspect.signature(verify_test_case))
print("verify_programs:", inspect.signature(verify_programs))
print("evaluate_candidate:", inspect.signature(evaluate_candidate))

verify_test_case: (original_code, original_language, generated_code, generated_language, input_data='', test_number=1)
verify_programs: (original_code, original_language, generated_code, generated_language, test_inputs)
evaluate_candidate: (candidate, original_code, original_language, test_inputs, benchmark_input)


In [384]:
def create_test_cases(test_inputs):
    """
    Normalize user/test inputs into a list of test cases.
    """

    if test_inputs is None:
        return [""]

    if isinstance(test_inputs, str):
        return [test_inputs]

    return list(test_inputs)

In [385]:
test_cases = create_test_cases([
    "",
    "1",
    "10",
    "100"
])

print(test_cases)
print("Number of tests:", len(test_cases))

['', '1', '10', '100']
Number of tests: 4


In [386]:
print(create_test_cases(""))
print(create_test_cases(None))

['']
['']


In [387]:
def run_multi_test_verification(
    original_code,
    original_language,
    generated_code,
    generated_language,
    test_inputs
):
    """
    Run verification across multiple test cases.
    """

    test_cases = create_test_cases(test_inputs)

    verification = verify_programs(
        original_code,
        original_language,
        generated_code,
        generated_language,
        test_cases
    )

    return verification

In [388]:
verification = run_multi_test_verification(
    test_code,
    "python",
    generated,
    "cpp",
    [
        "",
        "",
        ""
    ]
)

print("Verified:", verification.verified)
print("Total tests:", verification.total_tests)
print("Passed:", verification.passed_tests)
print("Failed:", verification.failed_tests)
print("Reason:", verification.reason)

Verified: True
Total tests: 3
Passed: 3
Failed: 0
Reason: All test cases passed.


In [389]:
def render_verification_report(verification):
    """
    Render verification results for the OptiForge UI.
    """

    if verification is None:
        return "### 🧪 Verification\n\nNot tested yet."

    status = "✅ VERIFIED" if verification.verified else "❌ REJECTED"

    lines = [
        "### 🧪 Verification",
        "",
        f"**Overall:** {status}",
        "",
        f"**Tests:** {verification.passed_tests} / {verification.total_tests} passed",
        ""
    ]

    for test in verification.test_results:
        result = "✅ PASS" if test.passed else "❌ FAIL"

        lines.append(
            f"- **Test {test.test_number}:** {result}"
        )

        if not test.passed and test.error:
            lines.append(
                f"  - Error: `{test.error}`"
            )

    if verification.reason:
        lines.extend([
            "",
            f"**Reason:** {verification.reason}"
        ])

    return "\n".join(lines)

In [390]:
print(render_verification_report(verification))

### 🧪 Verification

**Overall:** ✅ VERIFIED

**Tests:** 3 / 3 passed

- **Test 1:** ✅ PASS
- **Test 2:** ✅ PASS
- **Test 3:** ✅ PASS

**Reason:** All test cases passed.


In [391]:
import inspect

print(inspect.getsource(evaluate_candidate))

def evaluate_candidate(
    candidate,
    original_code,
    original_language,
    test_inputs,
    benchmark_input
):
    # Compilation/execution test
    initial_result = run_code(
        candidate.source_code,
        candidate.language,
        benchmark_input
    )

    if not initial_result.success:
        candidate.status = "compilation_failed"
        return candidate

    candidate.compiled = True

    # Correctness
    verification = verify_programs(
        original_code=original_code,
        original_language=original_language,
        generated_code=candidate.source_code,
        generated_language=candidate.language,
        test_inputs=test_inputs
    )

    candidate.verified = verification.verified

    if not candidate.verified:
        candidate.status = "verification_failed"
        return candidate

    # Benchmark
    benchmark = benchmark_comparison(
        original_code=original_code,
        original_language=original_language,
        generated_code=cand

In [400]:
candidate = CandidateResult(
    candidate_id="TEST-1",
    model="qwen/qwen3-coder-30b-a3b-instruct",
    source_code=generated,
    language="cpp"
)

candidate = evaluate_candidate(
    candidate,
    test_code,
    "python",
    ["", "", ""],
    ""
)

print("Compiled:", candidate.compiled)
print("Verified:", candidate.verified)
print("Status:", candidate.status)
print("Benchmark available:", candidate.benchmark is not None)
print("Verification:", candidate.verification)

Compiled: True
Verified: True
Status: verified
Benchmark available: True
Verification: VerificationResult(verified=True, total_tests=3, passed_tests=3, failed_tests=0, test_results=[TestCaseResult(test_number=1, passed=True, expected_output='500000500000\n', actual_output='500000500000\n', original_runtime=0.23987819999456406, generated_runtime=0.10455130005721003, error=''), TestCaseResult(test_number=2, passed=True, expected_output='500000500000\n', actual_output='500000500000\n', original_runtime=0.2635660000378266, generated_runtime=0.12795350002124906, error=''), TestCaseResult(test_number=3, passed=True, expected_output='500000500000\n', actual_output='500000500000\n', original_runtime=0.24305630009621382, generated_runtime=0.1576102999970317, error='')], reason='All test cases passed.')


In [401]:
wrong_candidate = CandidateResult(
    candidate_id="WRONG-1",
    model="test-model",
    source_code="""
#include <iostream>

int main() {
    std::cout << 123 << std::endl;
    return 0;
}
""",
    language="cpp"
)

wrong_candidate = evaluate_candidate(
    wrong_candidate,
    test_code,
    "python",
    ["", "", ""],
    ""
)

print("Compiled:", wrong_candidate.compiled)
print("Verified:", wrong_candidate.verified)
print("Status:", wrong_candidate.status)
print("Benchmark available:", wrong_candidate.benchmark is not None)

Compiled: True
Verified: False
Status: verification_failed
Benchmark available: False


In [402]:
print(candidate.verification)
print(render_verification_report(candidate.verification))

VerificationResult(verified=True, total_tests=3, passed_tests=3, failed_tests=0, test_results=[TestCaseResult(test_number=1, passed=True, expected_output='500000500000\n', actual_output='500000500000\n', original_runtime=0.23987819999456406, generated_runtime=0.10455130005721003, error=''), TestCaseResult(test_number=2, passed=True, expected_output='500000500000\n', actual_output='500000500000\n', original_runtime=0.2635660000378266, generated_runtime=0.12795350002124906, error=''), TestCaseResult(test_number=3, passed=True, expected_output='500000500000\n', actual_output='500000500000\n', original_runtime=0.24305630009621382, generated_runtime=0.1576102999970317, error='')], reason='All test cases passed.')
### 🧪 Verification

**Overall:** ✅ VERIFIED

**Tests:** 3 / 3 passed

- **Test 1:** ✅ PASS
- **Test 2:** ✅ PASS
- **Test 3:** ✅ PASS

**Reason:** All test cases passed.


In [405]:
def optimize_ui_with_verification(
    source_code,
    source_language,
    target_language,
    model_display_name
):
    # Run the existing pipeline
    generated, comparison, algorithm, insights, hotspots, recommendation = optimize_ui(
        source_code,
        source_language,
        target_language,
        model_display_name
    )

    # Find the stored candidate
    result_key = make_result_key(
        target_language,
        model_display_name
    )

    stored_result = optiforge_state["results"].get(result_key)

    # Render verification
    if stored_result is not None:
        candidate = stored_result["candidate"]

        if candidate.verification is not None:
            verification_report = render_verification_report(
                candidate.verification
            )
        else:
            verification_report = (
                "### 🧪 Verification\n\n"
                "Verification data unavailable."
            )
    else:
        verification_report = (
            "### 🧪 Verification\n\n"
            "No verified candidate available."
        )

    return (
        generated,
        comparison,
        algorithm,
        verification_report,
        insights,
        hotspots,
        recommendation
    )

In [408]:
generated, comparison, algorithm, verification, insights, hotspots, recommendation = optimize_ui_with_verification(
    test_code,
    "python",
    "cpp",
    "Qwen3 Coder 30B (OpenRouter)"
)

print(verification)

### 🧪 Verification

**Overall:** ✅ VERIFIED

**Tests:** 1 / 1 passed

- **Test 1:** ✅ PASS

**Reason:** All test cases passed.


In [407]:
app.close()

Closing server running on port: 7866


In [ ]:
app.launch(
    share=True
)

* Running on local URL:  http://127.0.0.1:7866

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


⚠️ Candidate 1 failed: 'NoneType' object has no attribute 'strip'
⚠️ Candidate 1 failed: 'NoneType' object has no attribute 'strip'
